# OZON E-CUP 2026 — Search LTV: разведочный анализ

**Задача.** Предсказать суммарный GMV пользователя (Поиск + Каталог) за 30 дней
**2026-02-14 … 2026-03-15** по истории **2025-01-01 … 2026-02-13**.
Метрика — **RMSLE** с `log1p`, 250 000 пользователей, public/private = 20/80.

Ноутбук отвечает на вопросы, от которых зависит архитектура решения:

| § | Вопрос |
|---|---|
| 1 | Что вообще в данных: инварианты, аномалии, сколько это стоит в RAM |
| 2 | Как выглядит глобальный таймлайн, есть ли дрейф/промо/структурные сдвиги перед таргет-окном |
| 3 | **Разреженность**: насколько прорежены ряды, как распределены дыры + **как отобрана выборка** (§3.1) |
| 4 | Как устроен таргет, что оптимизирует RMSLE и **что лежит в `sample_submit.csv`** (§4.1) |
| 5 | **Многооконный сэмплинг** (схема победителя GA Revenue): сколько якорей, есть ли дрейф между ними |
| 6 | Признаки: RFM + оконные агрегации на якорь |
| 7 | Корреляции: с таргетом и между собой (что дублируется) |
| 8 | **Hurdle-разложение**: `P(y>0)` × `E[y \| y>0]` — что чем предсказывается |
| 9 | **Фурье / спектр**: есть ли недельная структура и переносится ли она в признаки |
| 10 | Токенизация дневного поведения + матрица переходов (мостик к seq-моделям) |
| 11 | **Causal inference**: event-study вокруг промо, propensity-score/IPW, Granger-стиль предсказательная причинность |
| 12 | Выводы и что из этого делать |

> Ноутбук **только читает** `data_start/train.parquet`. Всё тяжёлое кешируется в `data_work/`.

## 0. Настройка

Машина — 18 GB RAM, поэтому дисциплина простая:
* исходник в `Int64/Float64` весит **~4.1 GB** в памяти — сразу даункастим до `~1.0 GB`;
* вместо дат работаем с целочисленным индексом дня `d = (event_date - 2025-01-01).days ∈ [0, 408]`
  — все оконные условия становятся сравнением `Int16`, это в разы быстрее и не плодит копий;
* всё, что дороже пары секунд, пишется в `data_work/` и переиспользуется.

In [1]:
import gc, json, math, warnings
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(24)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(60)

# --- корень проекта ищем вверх по дереву, чтобы ноутбук работал из любой папки
ROOT = Path.cwd()
while not (ROOT / "data_start").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_START = ROOT / "data_start"
DATA_WORK = ROOT / "data_work"
DATA_WORK.mkdir(exist_ok=True)

DAY0 = date(2025, 1, 1)          # d = 0
DAY_LAST = date(2026, 2, 13)     # d = 408, последний наблюдаемый день
N_DAYS = (DAY_LAST - DAY0).days + 1
HORIZON = 30                     # длина таргет-окна
N_USERS_EXPECTED = 250_000

# Пороги для тяжёлых секций: уменьшите, если мало RAM/времени.
SAMPLE_USERS = 60_000            # для спектра / раскладок по пользователям
RNG = np.random.default_rng(42)

print(f"ROOT       : {ROOT}")
print(f"дней всего : {N_DAYS}  ({DAY0} .. {DAY_LAST})")

ROOT       : /Users/kamiliskhakov/VS Code Project/OZON-ECUP-2026
дней всего : 409  (2025-01-01 .. 2026-02-13)


In [2]:
# ------------------------------------------------------------------ палитра
# Валидированная категориальная палитра (фиксированный порядок слотов, не цикл).
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
          "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ_BLUE = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
            "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
            "#184f95", "#104281", "#0d366b"]
SEQ = [[i / (len(SEQ_BLUE) - 1), c] for i, c in enumerate(SEQ_BLUE)]
DIVERGING = [[0.0, "#0d366b"], [0.25, "#5598e7"], [0.5, "#f0efec"],
             [0.75, "#e34948"], [1.0, "#8f1f1f"]]
STATUS = {"good": "#0ca30c", "warning": "#fab219",
          "serious": "#ec835a", "critical": "#d03b3b"}

SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"

FIG_W, FIG_H = 960, 580   # пропорции под слайд, а не под ширину экрана

pio.templates["ecup"] = go.layout.Template(
    layout=dict(
        colorway=SERIES,
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif",
                  size=13, color=INK2),
        title=dict(font=dict(size=16, color=INK), x=0.0, xanchor="left",
                   pad=dict(b=14)),
        margin=dict(l=64, r=28, t=64, b=120),
        xaxis=dict(gridcolor=GRID, zerolinecolor=AXIS, linecolor=AXIS,
                   tickfont=dict(color=MUTED, size=12), ticks="outside",
                   ticklen=4, tickcolor=AXIS, showline=True, automargin=True),
        yaxis=dict(gridcolor=GRID, zerolinecolor=AXIS, linecolor=AXIS,
                   tickfont=dict(color=MUTED, size=12), showline=False,
                   automargin=True),
        legend=dict(orientation="h", yanchor="top", y=-0.18, x=0, xanchor="left",
                    font=dict(size=12, color=INK2), bgcolor="rgba(0,0,0,0)"),
        hovermode="x unified",
        colorscale=dict(sequential=SEQ, diverging=DIVERGING),
    )
)
pio.templates.default = "ecup"


def show(fig, title=None, h=None, sub=None, w=None):
    """Единый финиш: заголовок, подпись, размер. automargin не даёт срезать подписи."""
    if title:
        t = title if sub is None else (
            f"{title}<br><span style='font-size:12px;color:{MUTED}'>{sub}</span>")
        fig.update_layout(title_text=t)
        if sub:
            fig.update_layout(margin=dict(t=84))
    fig.update_xaxes(automargin=True)
    fig.update_yaxes(automargin=True)
    if fig.layout.showlegend is False:          # без легенды нижнее поле не нужно
        fig.update_layout(margin=dict(b=60))
    fig.update_layout(height=h or FIG_H, width=w or FIG_W, autosize=False)
    fig.show()
    # ничего не возвращаем: иначе фигура отрисуется второй раз как результат ячейки


def line(fig, x, y, name, slot=0, width=2, dash=None, **kw):
    fig.add_trace(go.Scatter(
        x=x, y=y, name=name, mode="lines",
        line=dict(color=SERIES[slot % 8], width=width, dash=dash), **kw))
    return fig


def bars(x, y, slot=0, name="", text=None, orientation="v"):
    return go.Bar(x=x, y=y, name=name, orientation=orientation,
                  marker=dict(color=SERIES[slot % 8], cornerradius=4,
                              line=dict(color=SURFACE, width=1)),
                  text=text, textposition="outside",
                  textfont=dict(color=INK2, size=11), cliponaxis=False)


def hist_trace(values, bins=60, slot=0, name="", log_x=False, density=False):
    """Гистограмма считается numpy заранее — в фигуру уезжают только бины."""
    v = np.asarray(values, dtype="float64")
    v = v[np.isfinite(v)]
    if log_x:
        v = np.log10(np.clip(v, 1e-9, None))
    cnt, edges = np.histogram(v, bins=bins, density=density)
    ctr = (edges[:-1] + edges[1:]) / 2
    return go.Bar(x=ctr, y=cnt, name=name,
                  marker=dict(color=SERIES[slot % 8],
                              line=dict(color=SURFACE, width=0.5)),
                  hovertemplate="%{x:.3f} → %{y:,}<extra></extra>")


def ecdf(values, n=2000):
    v = np.sort(np.asarray(values, dtype="float64"))
    q = np.linspace(0, 1, min(n, len(v)))
    return np.quantile(v, q), q


def heat(z, x, y, colorscale=SEQ, zmid=None, hover="%{z:.3f}", zmin=None, zmax=None):
    return go.Heatmap(z=z, x=x, y=y, colorscale=colorscale, zmid=zmid,
                      zmin=zmin, zmax=zmax,
                      hovertemplate="x=%{x}<br>y=%{y}<br>" + hover + "<extra></extra>",
                      colorbar=dict(thickness=12, outlinewidth=0, len=0.85,
                                    tickfont=dict(color=MUTED, size=11)))


def fmt(n):
    for u, d in (("B", 1e9), ("M", 1e6), ("K", 1e3)):
        if abs(n) >= d:
            return f"{n/d:,.2f}{u}"
    return f"{n:,.2f}"


print("палитра и шаблон готовы")

палитра и шаблон готовы


## 1. Аудит данных

Проверяем то, на что потом будем опираться молча: типы, пропуски, дубли ключа,
арифметические инварианты между колонками и «странные» строки.

In [3]:
SCHEMA_CAST = {
    "user_id": pl.UInt32,
    "search": pl.Int8, "cat": pl.Int8,
    "has_search_to_cart": pl.Int8, "has_search_to_ord": pl.Int8,
    "has_cat_to_cart": pl.Int8, "has_cat_to_ord": pl.Int8,
    "search_to_cart": pl.UInt16, "search_to_ord": pl.UInt16,
    "cat_to_cart": pl.UInt16, "cat_to_ord": pl.UInt16,
    "to_cart": pl.UInt16, "to_ord": pl.UInt16,
    "searches": pl.UInt16,
    "gmv_search": pl.Float32, "gmv_cat": pl.Float32, "gmv": pl.Float32,
}
CACHE = DATA_WORK / "train_compact.parquet"

if CACHE.exists():
    df = pl.read_parquet(CACHE)
    print(f"кеш: {CACHE.name}")
else:
    raw = pl.read_parquet(DATA_START / "train.parquet")
    mem_raw = raw.estimated_size("mb")
    df = (
        raw.with_columns([pl.col(c).cast(t) for c, t in SCHEMA_CAST.items()])
           .with_columns(
               d=((pl.col("event_date") - pl.lit(DAY0)).dt.total_days()).cast(pl.Int16),
               dow=(pl.col("event_date").dt.weekday() - 1).cast(pl.Int8),  # 0=Пн
           )
           .drop("event_date")
           .sort(["user_id", "d"])
    )
    del raw; gc.collect()
    df.write_parquet(CACHE, compression="zstd")
    print(f"даункаст: {mem_raw:,.0f} MB → {df.estimated_size('mb'):,.0f} MB "
          f"({mem_raw/df.estimated_size('mb'):.1f}×)")

print(f"строк {df.height:,}   колонок {df.width}   RAM {df.estimated_size('mb'):,.0f} MB")
df.head(5)

кеш: train_compact.parquet
строк 30,631,006   колонок 19   RAM 1,139 MB


user_id,search,cat,has_search_to_cart,has_search_to_ord,has_cat_to_cart,has_cat_to_ord,search_to_cart,search_to_ord,cat_to_cart,cat_to_ord,gmv_search,gmv_cat,to_cart,to_ord,gmv,searches,d,dow
u32,i8,i8,i8,i8,i8,i8,u16,u16,u16,u16,f32,f32,u16,u16,f32,u16,i16,i8
2,1,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0.0,2,166,0
2,1,0,1,0,0,0,1,0,0,0,0.0,0.0,1,0,0.0,1,172,6
2,1,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0.0,1,219,4
2,1,0,1,0,0,0,1,0,0,0,0.0,0.0,1,0,0.0,1,229,0
2,1,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0.0,1,234,5


In [4]:
USERS = df["user_id"].unique().sort()
N_USERS = USERS.len()

audit = {
    "строк":                      df.height,
    "пользователей":              N_USERS,
    "дней в истории":             N_DAYS,
    "полная сетка user×day":      N_USERS * N_DAYS,
    "заполненность сетки, %":     round(100 * df.height / (N_USERS * N_DAYS), 2),
    "дублей ключа (user, d)":     df.height - df.select(["user_id", "d"]).n_unique(),
    "пропусков всего":            int(sum(df.null_count().row(0))),
}
for k, v in audit.items():
    print(f"{k:<28} {v:>15,}" if isinstance(v, int) else f"{k:<28} {v:>15}")

# --- инварианты: сходятся ли частные колонки в суммарные
inv = df.select(
    gmv_split=(pl.col("gmv") - pl.col("gmv_search") - pl.col("gmv_cat")).abs().max(),
    cart_split=(pl.col("to_cart").cast(pl.Int32) - pl.col("search_to_cart").cast(pl.Int32)
                - pl.col("cat_to_cart").cast(pl.Int32)).abs().max(),
    ord_split=(pl.col("to_ord").cast(pl.Int32) - pl.col("search_to_ord").cast(pl.Int32)
               - pl.col("cat_to_ord").cast(pl.Int32)).abs().max(),
    searches_wo_search=((pl.col("searches") > 0) & (pl.col("search") == 0)).sum(),
    ord_wo_flag=((pl.col("to_ord") > 0) & (pl.col("has_search_to_ord") == 0)
                 & (pl.col("has_cat_to_ord") == 0)).sum(),
    gmv_wo_ord=((pl.col("gmv") > 0) & (pl.col("to_ord") == 0)).sum(),
    ord_wo_gmv=((pl.col("to_ord") > 0) & (pl.col("gmv") == 0)).sum(),
)
print("\nинварианты:")
print(inv)

строк                             30,631,006
пользователей                        250,000
дней в истории                           409
полная сетка user×day            102,250,000
заполненность сетки, %                 29.96
дублей ключа (user, d)                     0
пропусков всего                            0

инварианты:
shape: (1, 7)
┌───────────┬────────────┬───────────┬────────────────────┬─────────────┬────────────┬────────────┐
│ gmv_split ┆ cart_split ┆ ord_split ┆ searches_wo_search ┆ ord_wo_flag ┆ gmv_wo_ord ┆ ord_wo_gmv │
│ ---       ┆ ---        ┆ ---       ┆ ---                ┆ ---         ┆ ---        ┆ ---        │
│ f32       ┆ i32        ┆ i32       ┆ u32                ┆ u32         ┆ u32        ┆ u32        │
╞═══════════╪════════════╪═══════════╪════════════════════╪═════════════╪════════════╪════════════╡
│ 0.001129  ┆ 0          ┆ 0         ┆ 0                  ┆ 0           ┆ 0          ┆ 0          │
└───────────┴────────────┴───────────┴────────────────────

**Что важно зафиксировать:**

* `gmv = gmv_search + gmv_cat`, `to_cart`/`to_ord` тоже точно раскладываются по каналам —
  значит суммарные колонки **линейно зависимы** от частных, и в модель их надо
  подавать осознанно (либо частные, либо суммарные + доли).
* Ключ `(user_id, d)` уникален → это честная дневная панель, просто прореженная.
* `searches > 0` ⇒ `search = 1` — флаг канала консистентен со счётчиком.

In [5]:
# --- «фантомные» строки: ни Поиска, ни Каталога
phantom = df.filter((pl.col("search") == 0) & (pl.col("cat") == 0))
print(f"строк без обоих каналов: {phantom.height:,} ({100*phantom.height/df.height:.1f}% всех строк)")
print(phantom.select(
    доля_с_корзиной=(pl.col("to_cart") > 0).mean(),
    доля_с_заказом=(pl.col("to_ord") > 0).mean(),
    доля_с_gmv=(pl.col("gmv") > 0).mean(),
    средний_gmv=pl.col("gmv").mean(),
    суммарный_gmv=pl.col("gmv").sum(),
))
print(f"\nдоля фантомного GMV во всём GMV: "
      f"{100*phantom['gmv'].sum()/df['gmv'].sum():.2f}%")

ph_ts = (phantom.group_by("d").agg(n=pl.len())
         .join(df.group_by("d").agg(tot=pl.len()), on="d").sort("d"))
fig = go.Figure()
line(fig, [DAY0 + timedelta(days=int(x)) for x in ph_ts["d"]],
     (ph_ts["n"] / ph_ts["tot"] * 100).to_numpy(), "доля фантомных строк, %", slot=1)
show(fig, "Строки без Поиска и без Каталога",
     sub="канал не проставлен, но активность есть",
     h=520)
del phantom; _ = gc.collect()

строк без обоих каналов: 4,667,981 (15.2% всех строк)
shape: (1, 5)
┌─────────────────┬────────────────┬────────────┬─────────────┬───────────────┐
│ доля_с_корзиной ┆ доля_с_заказом ┆ доля_с_gmv ┆ средний_gmv ┆ суммарный_gmv │
│ ---             ┆ ---            ┆ ---        ┆ ---         ┆ ---           │
│ f64             ┆ f64            ┆ f64        ┆ f32         ┆ f32           │
╞═════════════════╪════════════════╪════════════╪═════════════╪═══════════════╡
│ 0.025332        ┆ 0.007138       ┆ 0.007138   ┆ 0.315778    ┆ 1.4740e6      │
└─────────────────┴────────────────┴────────────┴─────────────┴───────────────┘

доля фантомного GMV во всём GMV: 0.54%


Эти строки — **не мусор**: часть из них несёт корзину и даже GMV (скорее всего отложенная
конверсия/заказ вне сессии Поиска и Каталога). Решение о том, считать ли такой день
«активным», меняет recency-признаки для заметной доли пользователей — ниже проверим оба
определения активности.

In [6]:
q = [0.5, 0.9, 0.99, 0.999, 0.9999, 1.0]
rows = []
for c in ["gmv", "gmv_search", "gmv_cat", "to_cart", "to_ord", "searches"]:
    v = df[c].to_numpy().astype("float64")
    nz = v[v > 0]
    rows.append(dict(колонка=c,
                     доля_нулей=float((v == 0).mean()),
                     **{f"q{int(p*10000)/100}_>0": float(np.quantile(nz, p)) for p in q}))
pl.DataFrame(rows)

колонка,доля_нулей,q50.0_>0,q90.0_>0,q99.0_>0,q99.9_>0,q99.99_>0,q100.0_>0
str,f64,f64,f64,f64,f64,f64,f64
"""gmv""",0.845356,23.332205,118.572122,557.202675,1769.741535,4436.288212,73830.296875
"""gmv_search""",0.852632,22.870112,115.189769,537.764808,1712.868384,4262.419093,73830.296875
"""gmv_cat""",0.989178,22.324816,122.177952,635.677372,2062.959216,5236.082617,18838.675781
"""to_cart""",0.630608,2.0,6.0,16.0,35.0,76.0,1535.0
"""to_ord""",0.845356,1.0,3.0,8.0,18.0,31.0,138.0
"""searches""",0.193419,2.0,9.0,25.0,58.0,113.0,630.0


Хвосты тяжёлые: между 99-м и 100-м перцентилем `gmv` разница в порядки.
Для RMSLE это скорее хорошо — `log1p` сжимает хвост, — но при обучении в исходной
шкале выбросы утащат MSE-лосс. **Вывод: учиться нужно в `log1p`-пространстве.**

## 2. Глобальный таймлайн

Смотрим, стабильна ли площадка. Любой структурный сдвиг в последние 60 дней истории —
прямая угроза: модель обучается на прошлом, а предсказывает 14.02–15.03.2026.

In [7]:
daily = (
    df.group_by("d").agg(
        dau=pl.len(),
        gmv=pl.col("gmv").sum(),
        gmv_search=pl.col("gmv_search").sum(),
        gmv_cat=pl.col("gmv_cat").sum(),
        orders=pl.col("to_ord").sum(),
        carts=pl.col("to_cart").sum(),
        searches=pl.col("searches").sum(),
        buyers=(pl.col("to_ord") > 0).sum(),
        n_search=pl.col("search").sum(),
        n_cat=pl.col("cat").sum(),
    )
    .sort("d")
    .with_columns(
        date=pl.col("d").map_elements(lambda x: DAY0 + timedelta(days=int(x)),
                                      return_dtype=pl.Date),
        dow=((pl.col("d") + 2) % 7).cast(pl.Int8),   # 2025-01-01 = среда (dow=2)
    )
)
DATES = daily["date"].to_list()
daily.head(3)

d,dau,gmv,gmv_search,gmv_cat,orders,carts,searches,buyers,n_search,n_cat,date,dow
i16,u32,f32,f32,f32,i64,i64,i64,u32,i64,i64,date,i8
0,43516,374066.5,335724.78125,38341.730469,8827,38548,136443,5620,36918,8010,2025-01-01,2
1,52556,489228.03125,443611.21875,45616.847656,12381,51779,174063,7394,44387,10571,2025-01-02,3
2,53609,473332.46875,434329.96875,39002.515625,12340,51388,175456,7487,45432,10414,2025-01-03,4


In [8]:
metrics = [("dau", "Активных пользователей", 0),
           ("gmv", "GMV", 1),
           ("orders", "Заказанных товаров", 2),
           ("carts", "Добавлений в корзину", 6),
           ("searches", "Поисковых запросов", 4)]

fig = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                    vertical_spacing=0.035,
                    subplot_titles=[m[1] for m in metrics])
for i, (col, name, slot) in enumerate(metrics, start=1):
    y = daily[col].to_numpy().astype("float64")
    fig.add_trace(go.Scatter(x=DATES, y=y, name=name, mode="lines",
                             line=dict(color=SERIES[slot], width=1.4),
                             showlegend=False), row=i, col=1)
    # скользящее среднее 7д — снимает недельную пилу
    ma = np.convolve(y, np.ones(7) / 7, mode="same")
    ma[:3] = np.nan; ma[-3:] = np.nan
    fig.add_trace(go.Scatter(x=DATES, y=ma, mode="lines", showlegend=False,
                             line=dict(color=INK, width=2)), row=i, col=1)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
    a.x, a.xanchor = 0, "left"
show(fig, "Дневная динамика площадки",
     sub="толстая линия — MA(7)", h=980, w=1060)

In [9]:
# нормировка на активного пользователя убирает эффект «людей стало больше/меньше»
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["GMV на активного пользователя",
                                    "Доля покупающих среди активных, %"])
per_dau = (daily["gmv"] / daily["dau"]).to_numpy()
conv = (daily["buyers"] / daily["dau"] * 100).to_numpy()
for i, (y, slot) in enumerate([(per_dau, 1), (conv, 2)], start=1):
    fig.add_trace(go.Scatter(x=DATES, y=y, mode="lines", showlegend=False,
                             line=dict(color=SERIES[slot], width=1.2)), row=i, col=1)
    ma = np.convolve(y, np.ones(7) / 7, mode="same"); ma[:3] = np.nan; ma[-3:] = np.nan
    fig.add_trace(go.Scatter(x=DATES, y=ma, mode="lines", showlegend=False,
                             line=dict(color=INK, width=2)), row=i, col=1)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2); a.x, a.xanchor = 0, "left"
show(fig, "Интенсивность на пользователя", h=560)

In [10]:
# --- доля каналов в GMV
fig = go.Figure()
share_s = (daily["gmv_search"] / daily["gmv"] * 100).to_numpy()
ma_s = np.convolve(share_s, np.ones(7) / 7, mode="same"); ma_s[:3] = np.nan; ma_s[-3:] = np.nan
line(fig, DATES, share_s, "Поиск, % GMV", slot=0, width=1.2, opacity=0.45)
line(fig, DATES, ma_s, "Поиск, MA(7)", slot=0, width=2.5)
line(fig, DATES, 100 - ma_s, "Каталог, MA(7)", slot=1, width=2.5)
fig.update_yaxes(title="% GMV", range=[0, 100])
fig.add_hline(y=50, line=dict(color=AXIS, width=1, dash="dot"))
show(fig, "Как GMV делится между Поиском и Каталогом", h=560)

In [11]:
# --- календарная теплокарта: неделя × день недели
DOW_RU = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
cal = daily.with_columns(week=((pl.col("d") + 2) // 7).cast(pl.Int32))  # 2025-01-01 = Ср

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=["DAU", "GMV"])
for i, col in enumerate(["dau", "gmv"], start=1):
    p = cal.pivot(values=col, index="dow", on="week", aggregate_function="first").sort("dow")
    weeks = [c for c in p.columns if c != "dow"]
    z = p.select(weeks).to_numpy().astype("float64")
    xs = [DAY0 + timedelta(days=int(w) * 7 - 2) for w in map(int, weeks)]
    fig.add_trace(heat(z, xs, DOW_RU, hover="%{z:,.0f}"), row=i, col=1)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2); a.x, a.xanchor = 0, "left"
fig.update_traces(colorbar=dict(len=0.42, thickness=11, outlinewidth=0,
                                tickfont=dict(color=MUTED, size=10)))
fig.data[0].colorbar.y, fig.data[1].colorbar.y = 0.79, 0.21
show(fig, "Календарная теплокарта: неделя × день недели",
     sub="тёмные столбцы — промо и праздники", h=560, w=1060)

In [12]:
dow_prof = (daily.group_by("dow").agg(
    dau=pl.col("dau").mean(), gmv=pl.col("gmv").mean(),
    gmv_per_dau=(pl.col("gmv").sum() / pl.col("dau").sum())).sort("dow"))
fig = make_subplots(rows=1, cols=3, subplot_titles=["DAU", "GMV", "GMV / DAU"],
                    horizontal_spacing=0.09)
for i, (c, slot) in enumerate([("dau", 0), ("gmv", 1), ("gmv_per_dau", 2)], start=1):
    v = dow_prof[c].to_numpy()
    rel = 100 * (v / v.mean() - 1)
    fig.add_trace(bars(DOW_RU, rel, slot=slot,
                       text=[f"{x:+.1f}%" for x in rel]), row=1, col=i)
    fig.update_yaxes(title="откл. от среднего, %" if i == 1 else None, row=1, col=i)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Профиль дня недели", h=520)

In [13]:
# --- топ-дни: промо-календарь, который потом пригодится в §11
top = daily.sort("gmv", descending=True).head(15).select(
    ["date", "dau", "gmv", "orders", "searches"]).with_columns(
    gmv_vs_median=(pl.col("gmv") / daily["gmv"].median()).round(2))
top

date,dau,gmv,orders,searches,gmv_vs_median
date,u32,f32,i64,i64,f32
2025-12-27,92960,1093028.5,30849,316828,1.65
2025-12-25,96169,1.0927e6,30896,317639,1.65
2025-12-21,101903,1.0665e6,32367,362860,1.61
2025-12-26,90912,1.0457e6,29320,300034,1.58
2025-12-28,93986,1.03124e6,30122,319270,1.56
2025-12-14,103617,1.0255e6,30743,373748,1.55
2025-12-22,99558,1.0252e6,29705,336698,1.55
2025-12-15,102272,1.0134e6,29514,350645,1.53
2025-12-23,97679,1.0117e6,29570,326365,1.53


Пиковые даты стоит выписать: они дают «естественные эксперименты» для §11 и объясняют,
почему сырое `gmv_30d` смещено, если в окно попал промо-день.

In [14]:
# --- месячная сводка + проверка на дрейф прямо перед таргет-окном
monthly = (daily.with_columns(month=pl.col("date").dt.strftime("%Y-%m"))
           .group_by("month").agg(
               дней=pl.len(), DAU_ср=pl.col("dau").mean().round(0),
               GMV=pl.col("gmv").sum(), заказы=pl.col("orders").sum(),
               GMV_на_DAU=(pl.col("gmv").sum() / pl.col("dau").sum()).round(3))
           .sort("month"))
monthly

month,дней,DAU_ср,GMV,заказы,GMV_на_DAU
str,u32,f64,f32,i64,f64
"""2025-01""",31,57107.0,1.4558424e7,402490,8.224
"""2025-02""",28,61792.0,1.4484722e7,416789,8.372
"""2025-03""",31,64186.0,1.7418118e7,489485,8.754
"""2025-04""",30,64915.0,1.7005648e7,474734,8.732
"""2025-05""",31,65440.0,1.775749e7,510083,8.753
"""2025-06""",30,67070.0,1.8715814e7,535242,9.302
"""2025-07""",31,69163.0,2.0441936e7,591382,9.534
"""2025-08""",31,73801.0,2.257182e7,671725,9.866
"""2025-09""",30,78090.0,2.1852364e7,642950,9.328


In [15]:
# сравниваем последние 30 дней истории с тем же окном годом/кварталом ранее
def win(a, b):
    return daily.filter(pl.col("d").is_between(a, b)).select(
        pl.col("gmv").sum(), pl.col("dau").mean(), pl.col("orders").sum()).row(0)

last = win(N_DAYS - 30, N_DAYS - 1)
prev = win(N_DAYS - 60, N_DAYS - 31)
yago = win(N_DAYS - 30 - 364, N_DAYS - 1 - 364)
tbl = pl.DataFrame({
    "окно": ["последние 30д", "предыдущие 30д", "год назад, те же 30д"],
    "GMV": [last[0], prev[0], yago[0]],
    "DAU_ср": [last[1], prev[1], yago[1]],
    "заказы": [last[2], prev[2], yago[2]],
}).with_columns(GMV_к_последнему=(pl.col("GMV") / last[0]).round(3))
tbl

окно,GMV,DAU_ср,заказы,GMV_к_последнему
str,f64,f64,i64,f64
"""последние 30д""",2.1008508e7,93132.233333,700059,1.0
"""предыдущие 30д""",2.5356486e7,90221.966667,760932,1.207
"""год назад, те же 30д""",1.4376082e7,59219.466667,406680,0.684


## 3. Разреженность — центральная особенность данных

Полная сетка `250 000 × 409 ≈ 102.3 млн` строк, в данных — `30.6 млн`.
То есть **~70 % ячеек отсутствуют**, и это не «пропуски», а явные нули активности.
Дозаполнять всё нулями нельзя (RAM), но нужно понимать *структуру дыр*:
одно дело равномерно редкий пользователь, другое — активный, который ушёл в декабре.

In [16]:
CACHE_U = DATA_WORK / "user_profile.parquet"
if CACHE_U.exists():
    up = pl.read_parquet(CACHE_U)
else:
    up = (
        df.group_by("user_id").agg(
            n_days=pl.len(),
            first_d=pl.col("d").min(),
            last_d=pl.col("d").max(),
            n_ord_days=(pl.col("to_ord") > 0).sum(),
            n_cart_days=(pl.col("to_cart") > 0).sum(),
            n_search_days=pl.col("search").sum(),
            n_cat_days=pl.col("cat").sum(),
            n_phantom_days=((pl.col("search") == 0) & (pl.col("cat") == 0)).sum(),
            gmv_total=pl.col("gmv").sum(),
            gmv_search=pl.col("gmv_search").sum(),
            gmv_cat=pl.col("gmv_cat").sum(),
            ord_total=pl.col("to_ord").sum(),
            cart_total=pl.col("to_cart").sum(),
            searches_total=pl.col("searches").sum(),
            last_ord_d=pl.when(pl.col("to_ord") > 0).then(pl.col("d")).max(),
        )
        .with_columns(
            span=(pl.col("last_d") - pl.col("first_d") + 1).cast(pl.Int32),
            recency=(N_DAYS - 1 - pl.col("last_d")).cast(pl.Int32),
            recency_ord=(N_DAYS - 1 - pl.col("last_ord_d")).cast(pl.Int32),
        )
        .with_columns(
            density_span=(pl.col("n_days") / pl.col("span")),
            density_all=(pl.col("n_days") / N_DAYS),
            buy_rate=(pl.col("n_ord_days") / pl.col("n_days")),
            aov=pl.when(pl.col("ord_total") > 0)
                  .then(pl.col("gmv_total") / pl.col("ord_total")),
        )
        .sort("user_id")
    )
    up.write_parquet(CACHE_U, compression="zstd")
print(up.shape)
up.head(3)

(250000, 23)


user_id,n_days,first_d,last_d,n_ord_days,n_cart_days,n_search_days,n_cat_days,n_phantom_days,gmv_total,…,cart_total,searches_total,last_ord_d,span,recency,recency_ord,density_span,density_all,buy_rate,aov
u32,u32,i16,i16,u32,u32,i64,i64,u32,f32,…,i64,i64,i16,i32,i32,i32,f64,f64,f64,f64
2,19,166,408,1,11,18,0,1,16.315287,…,20,46,374,243,0,34,0.078189,0.046455,0.052632,16.315287
7,113,3,408,29,35,99,9,11,1313.133667,…,60,304,408,406,0,0,0.278325,0.276284,0.256637,31.265087
15,80,1,401,5,35,64,6,14,2445.368164,…,85,112,374,401,7,34,0.199501,0.195599,0.0625,305.671021


In [17]:
grid = N_USERS * N_DAYS
print(f"полная сетка user×day        : {grid:,}")
print(f"фактических строк            : {df.height:,}")
print(f"заполненность                : {100*df.height/grid:.2f}%")
print(f"строк с gmv>0                : {(df['gmv']>0).sum():,} "
      f"({100*(df['gmv']>0).mean():.2f}% строк, {100*(df['gmv']>0).sum()/grid:.2f}% сетки)")
print()
print(f"пользователей без единой покупки за 409 дней : "
      f"{(up['ord_total']==0).sum():,} ({100*(up['ord_total']==0).mean():.1f}%)")
print(f"пользователей, не заходивших последние 30д   : "
      f"{(up['recency']>=30).sum():,} ({100*(up['recency']>=30).mean():.1f}%)")
print(f"пользователей, не покупавших последние 30д   : "
      f"{(up['recency_ord'].fill_null(9999)>=30).sum():,} "
      f"({100*(up['recency_ord'].fill_null(9999)>=30).mean():.1f}%)")

полная сетка user×day        : 102,250,000
фактических строк            : 30,631,006
заполненность                : 29.96%
строк с gmv>0                : 4,736,907 (15.46% строк, 4.63% сетки)

пользователей без единой покупки за 409 дней : 30,832 (12.3%)
пользователей, не заходивших последние 30д   : 0 (0.0%)
пользователей, не покупавших последние 30д   : 114,835 (45.9%)


In [18]:
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.075,
                    subplot_titles=["Активных дней на пользователя",
                                    "Плотность внутри своего периода жизни",
                                    "Дней с покупкой"])
fig.add_trace(hist_trace(up["n_days"].to_numpy(), bins=70, slot=0), row=1, col=1)
fig.add_trace(hist_trace(up["density_span"].to_numpy(), bins=70, slot=2), row=1, col=2)
fig.add_trace(hist_trace(up["n_ord_days"].to_numpy(), bins=70, slot=1), row=1, col=3)
fig.update_yaxes(type="log", title="пользователей", row=1, col=1)
fig.update_yaxes(type="log", row=1, col=2); fig.update_yaxes(type="log", row=1, col=3)
fig.update_xaxes(title="дней из 409", row=1, col=1)
fig.update_xaxes(title="active_days / span", row=1, col=2)
fig.update_xaxes(title="дней с to_ord>0", row=1, col=3)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
fig.update_layout(showlegend=False, hovermode="closest", bargap=0.02)
show(fig, "Насколько прорежен ряд одного пользователя", h=560)

In [19]:
# --- ECDF: сколько пользователей «почти пустые»
fig = go.Figure()
for i, (c, nm) in enumerate([("n_days", "активных дней"),
                             ("n_ord_days", "дней с покупкой"),
                             ("n_cart_days", "дней с корзиной")]):
    x, y = ecdf(up[c].to_numpy())
    line(fig, x, y * 100, nm, slot=i, width=2.5)
fig.update_xaxes(title="дней", type="log", range=[0, 2.62])
fig.update_yaxes(title="% пользователей ≤ x", range=[0, 100])
fig.update_layout(hovermode="x")
show(fig, "ECDF активности пользователя", h=560)

for p in [10, 25, 50, 75, 90, 99]:
    print(f"p{p:<3}: активных дней {np.percentile(up['n_days'], p):6.0f} | "
          f"дней с покупкой {np.percentile(up['n_ord_days'], p):5.0f} | "
          f"GMV всего {np.percentile(up['gmv_total'], p):10.1f}")

p10 : активных дней     26 | дней с покупкой     0 | GMV всего        0.0
p25 : активных дней     51 | дней с покупкой     3 | GMV всего       73.8
p50 : активных дней    102 | дней с покупкой    10 | GMV всего      388.2
p75 : активных дней    178 | дней с покупкой    26 | GMV всего     1210.4
p90 : активных дней    254 | дней с покупкой    49 | GMV всего     2764.4
p99 : активных дней    356 | дней с покупкой   116 | GMV всего     9680.5


In [20]:
# --- распределение дыр: интервалы между активными днями
SUB = RNG.choice(USERS.to_numpy(), size=min(40_000, N_USERS), replace=False)
sub = df.filter(pl.col("user_id").is_in(SUB)).select(["user_id", "d"]).sort(["user_id", "d"])
gaps = (sub.with_columns(gap=(pl.col("d") - pl.col("d").shift(1).over("user_id")))
          .drop_nulls("gap"))
g = gaps["gap"].to_numpy()
print(f"интервалов: {len(g):,}   медиана {np.median(g):.0f}д   "
      f"среднее {g.mean():.2f}д   p90 {np.percentile(g,90):.0f}д   max {g.max()}д")

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.09,
                    subplot_titles=["Распределение интервалов между активными днями",
                                    "Средний интервал по пользователю"])
cnt = np.bincount(g, minlength=40)[:40]
fig.add_trace(go.Bar(x=np.arange(40), y=cnt, marker=dict(color=SERIES[0], cornerradius=3,
              line=dict(color=SURFACE, width=0.5)), hovertemplate="gap=%{x}д → %{y:,}<extra></extra>"),
              row=1, col=1)
mg = gaps.group_by("user_id").agg(mean_gap=pl.col("gap").mean())["mean_gap"].to_numpy()
fig.add_trace(hist_trace(mg, bins=70, slot=2), row=1, col=2)
fig.update_yaxes(type="log", title="интервалов", row=1, col=1)
fig.update_yaxes(type="log", title="пользователей", row=1, col=2)
fig.update_xaxes(title="дней между визитами", row=1, col=1)
fig.update_xaxes(title="средний интервал, дней", type="log", row=1, col=2)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
fig.update_layout(showlegend=False, hovermode="closest", bargap=0.06)
show(fig, "Структура дыр в рядах", h=560)

интервалов: 4,835,900   медиана 1д   среднее 3.00д   p90 6д   max 338д


Форма распределения интервалов — почти геометрическая с тяжёлым хвостом. Это ровно та
ситуация, под которую заточены **BTYD-модели** (Pareto/NBD, BG/NBD): «пока жив — покупает
с частотой λ, в какой-то момент умирает». Стоит держать в голове как отдельный источник
признаков (`P(alive)`, `E[transactions]`), даже если основная модель — градиентный бустинг.

In [21]:
# --- растр активности: наглядно про «прорежённость»
n_show = 320
pick = up.sort("n_days", descending=True)
idx = np.linspace(0, N_USERS - 1, n_show).astype(int)     # срез по всему спектру активности
pick_ids = pick["user_id"].to_numpy()[idx]
raster_src = df.filter(pl.col("user_id").is_in(pick_ids)).select(["user_id", "d", "gmv"])
pos = {u: i for i, u in enumerate(pick_ids)}
Z = np.zeros((n_show, N_DAYS), dtype="float32")
uu = raster_src["user_id"].to_numpy(); dd = raster_src["d"].to_numpy()
gg = raster_src["gmv"].to_numpy()
ri = np.array([pos[u] for u in uu])
Z[ri, dd] = 1.0
Z[ri[gg > 0], dd[gg > 0]] = 2.0        # 0 нет активности, 1 визит, 2 визит с покупкой

fig = go.Figure(go.Heatmap(
    z=Z, x=DATES, colorscale=[[0.0, SURFACE], [0.49, SURFACE],
                              [0.5, "#9ec5f4"], [0.74, "#9ec5f4"],
                              [0.75, "#0d366b"], [1.0, "#0d366b"]],
    zmin=0, zmax=2, showscale=False,
    hovertemplate="%{x|%Y-%m-%d}<extra></extra>"))
# autorange="reversed": строка 0 — самый активный пользователь, ей место наверху
fig.update_yaxes(title="пользователи, сверху вниз по убыванию активности",
                 showticklabels=False, autorange="reversed")
show(fig, "Растр активности: 320 пользователей × 409 дней",
     sub="белое — визита нет, тёмное — визит с покупкой", h=620, w=1060)
del Z, raster_src; _ = gc.collect()

In [22]:
# --- когорты: когда пользователь появился и когда исчез
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.09,
                    subplot_titles=["Первый визит (first_d)", "Последний визит (last_d)"])
for i, (c, slot) in enumerate([("first_d", 0), ("last_d", 7)], start=1):
    v = up[c].to_numpy()
    cnt = np.bincount(v, minlength=N_DAYS)
    fig.add_trace(go.Scatter(x=DATES, y=cnt, mode="lines",
                             line=dict(color=SERIES[slot], width=1.6),
                             showlegend=False), row=1, col=i)
    fig.update_yaxes(type="log", title="пользователей" if i == 1 else None, row=1, col=i)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
show(fig, "Границы жизни пользователя в выборке",
     sub="пики на краях — когорты до и после наблюдений", h=520)

In [23]:
# --- retention: доля пользователей, активных в скользящем 30-дневном окне
act = df.select(["user_id", "d"])
roll = []
for a in range(29, N_DAYS, 7):
    n = act.filter(pl.col("d").is_between(a - 29, a))["user_id"].n_unique()
    roll.append((a, n))
ra = np.array(roll)
fig = go.Figure()
line(fig, [DAY0 + timedelta(days=int(x)) for x in ra[:, 0]], ra[:, 1] / N_USERS * 100,
     "активны в последние 30 дней, %", slot=0, width=2.5)
fig.update_yaxes(title="% от 250K", rangemode="tozero")
show(fig, "MAU как доля всей базы", h=520)

In [24]:
# --- когортная retention-теплокарта
coh = (up.with_columns(cohort=(pl.col("first_d") // 28).cast(pl.Int32))
         .select(["user_id", "cohort"]))
ev = (df.select(["user_id", "d"]).join(coh, on="user_id")
        .with_columns(period=(pl.col("d") // 28).cast(pl.Int32))
        .group_by(["cohort", "period"]).agg(n=pl.col("user_id").n_unique()))
size = coh.group_by("cohort").agg(base=pl.len())
ret = (ev.join(size, on="cohort")
         .with_columns(age=pl.col("period") - pl.col("cohort"),
                       rate=pl.col("n") / pl.col("base"))
         .filter(pl.col("age") >= 0))
p = ret.pivot(values="rate", index="cohort", on="age", aggregate_function="first").sort("cohort")
ages = sorted([int(c) for c in p.columns if c != "cohort"])
Zr = p.select([str(a) for a in ages]).to_numpy().astype("float64") * 100
coh_lbl = [(DAY0 + timedelta(days=int(c) * 28)).strftime("%Y-%m-%d") for c in p["cohort"]]
fig = go.Figure(heat(Zr, [f"+{a}" for a in ages], coh_lbl, hover="%{z:.1f}%"))
fig.update_xaxes(title="периодов по 28 дней от появления")
fig.update_yaxes(title="когорта (первый визит)", autorange="reversed")
show(fig, "Retention по когортам первого визита",
     sub="% когорты, активной в периоде", h=560, w=1040)

### 3.1 Как отобрана выборка пользователей — и почему это важнее всего остального

`recency` (дней с последней активности на 2026-02-13) не просто мал — у него **жёсткий
потолок**. Проверим, случайность это или правило отбора.

In [25]:
r = up["recency"].to_numpy()
print(f"максимальный recency по всей базе: {r.max()} дней")
print(f"пользователей с recency >= 30    : {(r >= 30).sum()}")

print("\nдоля пользователей, активных в 30-дневном окне, по календарным блокам:")
for A in range(N_DAYS - 1, 40, -30):
    a = df.filter(pl.col("d").is_between(A - 29, A))["user_id"].n_unique() / N_USERS
    mark = "  ← 100%" if a > 0.9999 else ""
    print(f"  [{DAY0+timedelta(days=A-29)} .. {DAY0+timedelta(days=A)}]: {a:.4f}{mark}")

print("\nто же окно, но со сдвигом на 5 дней (проверка, что 100% — не совпадение):")
for A in range(N_DAYS - 1, N_DAYS - 1 - 40, -5):
    a = df.filter(pl.col("d").is_between(A - 29, A))["user_id"].n_unique() / N_USERS
    print(f"  [d={A-29:3d} .. d={A:3d}]: {a:.4f}")

максимальный recency по всей базе: 29 дней
пользователей с recency >= 30    : 0

доля пользователей, активных в 30-дневном окне, по календарным блокам:
  [2026-01-15 .. 2026-02-13]: 1.0000  ← 100%
  [2025-12-16 .. 2026-01-14]: 1.0000  ← 100%
  [2025-11-16 .. 2025-12-15]: 1.0000  ← 100%
  [2025-10-17 .. 2025-11-15]: 0.9326
  [2025-09-17 .. 2025-10-16]: 0.9017
  [2025-08-18 .. 2025-09-16]: 0.8752
  [2025-07-19 .. 2025-08-17]: 0.8510
  [2025-06-19 .. 2025-07-18]: 0.8334


  [2025-05-20 .. 2025-06-18]: 0.8228
  [2025-04-20 .. 2025-05-19]: 0.8087
  [2025-03-21 .. 2025-04-19]: 0.7977
  [2025-02-19 .. 2025-03-20]: 0.7849
  [2025-01-20 .. 2025-02-18]: 0.7671

то же окно, но со сдвигом на 5 дней (проверка, что 100% — не совпадение):
  [d=379 .. d=408]: 1.0000
  [d=374 .. d=403]: 0.9900
  [d=369 .. d=398]: 0.9838
  [d=364 .. d=393]: 0.9795
  [d=359 .. d=388]: 0.9804


  [d=354 .. d=383]: 0.9874
  [d=349 .. d=378]: 1.0000
  [d=344 .. d=373]: 0.9901


**Правило отбора восстанавливается однозначно:** в выборку попали ровно те пользователи,
у кого есть хотя бы одно событие **в каждом** из трёх последних календарных 30-дневных
блоков (16.11–15.12, 16.12–14.01, 15.01–13.02). Сдвинутые окна дают 98–99 %, а эти три — ровно 100 %.

Последствия — прямые и жёсткие:

1. **«Мёртвых» пользователей в выборке нет.** Все 250 000 активны в последний месяц.
   Значит `recency ∈ [0, 29]` на боевом якоре, и рассуждения вида «половина базы ушла»
   к этой задаче неприменимы.
2. **Исторические якоря не сопоставимы с боевым «из коробки».** На якоре `d=288`
   правилу отбора удовлетворяют только ~90 % пользователей — остальные при обучении
   создают популяцию, которой в тесте не существует. Это классический covariate shift,
   и он бьёт ровно по калибровке `P(y>0)`.
3. **Лечится это одной строчкой:** на каждом обучающем якоре `A` оставлять только тех,
   кто прошёл бы тот же фильтр — активность в каждом из блоков
   `[A-89, A-60]`, `[A-59, A-30]`, `[A-29, A]`. Ниже это оформлено функцией и проверено.

In [26]:
def selected_users(anchor_d: int, blocks: int = 3, block: int = 30) -> pl.Series:
    """Пользователи, проходящие правило отбора организаторов на момент anchor_d."""
    lo = anchor_d - blocks * block + 1
    s = (df.filter(pl.col("d").is_between(lo, anchor_d))
           .with_columns(b=((pl.col("d") - lo) // block).cast(pl.Int8))
           .group_by("user_id").agg(nb=pl.col("b").n_unique()))
    return s.filter(pl.col("nb") == blocks)["user_id"]


chk = selected_users(N_DAYS - 1)
print(f"проверка на боевом якоре: правилу удовлетворяют {chk.len():,} из {N_USERS:,} "
      f"→ {'правило восстановлено верно' if chk.len() == N_USERS else 'НЕ СХОДИТСЯ'}")

sel_share = []
for A in range(N_DAYS - 1, 90, -30):
    n = selected_users(A).len()
    sel_share.append((A, n / N_USERS))
sa = np.array(sel_share)
fig = go.Figure()
line(fig, [DAY0 + timedelta(days=int(x)) for x in sa[:, 0]], sa[:, 1] * 100,
     "проходят фильтр отбора, %", slot=0, width=2.5)
fig.add_hline(y=100, line=dict(color=AXIS, width=1.5, dash="dash"))
fig.update_yaxes(title="% от 250K", range=[60, 105])
show(fig, "Сколько пользователей прошли бы отбор на историческом якоре", h=520)

проверка на боевом якоре: правилу удовлетворяют 250,000 из 250,000 → правило восстановлено верно


In [27]:
# --- концентрация GMV: кривая Лоренца и Джини
v = np.sort(up["gmv_total"].to_numpy().astype("float64"))
cum = np.cumsum(v); cum = cum / cum[-1]
frac = np.arange(1, len(v) + 1) / len(v)
gini = 1 - 2 * np.trapezoid(cum, frac)
step = max(1, len(v) // 3000)
fig = go.Figure()
line(fig, frac[::step] * 100, cum[::step] * 100, "фактическое распределение", slot=0, width=2.5)
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", name="равномерное",
                         line=dict(color=AXIS, width=1.5, dash="dash")))
fig.update_xaxes(title="% пользователей (по возрастанию GMV)")
fig.update_yaxes(title="% суммарного GMV")
show(fig, f"Кривая Лоренца по GMV — коэффициент Джини {gini:.3f}", h=560)

for p_ in [1, 5, 10, 20, 50]:
    k = int(len(v) * (1 - p_ / 100))
    print(f"топ-{p_:<2}% пользователей дают {100*(1-cum[k]):5.1f}% GMV")
print(f"пользователей с нулевым GMV за всю историю: "
      f"{(up['gmv_total']==0).sum():,} ({100*(up['gmv_total']==0).mean():.1f}%)")

топ-1 % пользователей дают  14.9% GMV
топ-5 % пользователей дают  37.3% GMV
топ-10% пользователей дают  53.0% GMV
топ-20% пользователей дают  71.9% GMV
топ-50% пользователей дают  94.8% GMV
пользователей с нулевым GMV за всю историю: 30,832 (12.3%)


### Вывод по §3

1. Заполненность сетки ~30 %, а «полезных» ячеек (`gmv>0`) — **менее 5 % от полной сетки**.
   Разреженность здесь не техническая деталь, а сама структура задачи ⇒ hurdle (§8).
2. Дыры структурны: типичный интервал между визитами — единицы дней, но хвост уходит
   в сотни. Это ровно та форма, под которую сделаны BTYD-модели.
3. **Выборка условна на недавней активности** (§3.1) — это меняет и трактовку recency,
   и схему кросс-валидации. У нас при этом **нет cooling period** (в GA Revenue было 46 дней),
   поэтому свежие признаки работают заметно сильнее, чем в том соревновании.
4. Концентрация GMV высокая (Джини считается выше): верхний дециль делает больше половины
   оборота. В `log1p`-шкале хвост сжимается, но калибровка на нём всё равно решает.

## 4. Таргет и метрика

Таргет для якоря `T`: `y = Σ gmv` по дням `(T, T+30]`.
Финальный якорь — `T = 2026-02-13` (`d = 408`), его таргет мы не видим.
Ближайший **полностью наблюдаемый** якорь — `d = 378` (`2026-01-14`), окно `2026-01-15 … 2026-02-13`.
Это наш основной валидационный якорь: он максимально похож на боевой.

RMSLE — это RMSE в пространстве `log1p`. Отсюда два следствия, которые определяют всё:
* оптимальный **константный** прогноз = `expm1(mean(log1p(y)))`, а не среднее `y`;
* оптимальный прогноз для объекта = `expm1(E[log1p(y) | x])` — модель должна
  предсказывать **среднее логарифма**, а не логарифм среднего. Обучаться нужно прямо в `log1p`.

In [28]:
def make_target(anchor_d: int, horizon: int = HORIZON) -> pl.DataFrame:
    """Σ gmv по (anchor_d, anchor_d + horizon] для всех пользователей, 0 для неактивных."""
    t = (df.filter(pl.col("d").is_between(anchor_d + 1, anchor_d + horizon))
           .group_by("user_id").agg(y=pl.col("gmv").sum().cast(pl.Float64)))
    return (pl.DataFrame({"user_id": USERS}).join(t, on="user_id", how="left")
              .with_columns(pl.col("y").fill_null(0.0)))


def rmsle(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred, dtype="float64"), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(np.asarray(y_true, "float64")))**2)))


ANCHOR_VAL = N_DAYS - 1 - HORIZON          # 378 → окно 2026-01-15 .. 2026-02-13
ANCHOR_FINAL = N_DAYS - 1                  # 408 → боевое окно 2026-02-14 .. 2026-03-15
print(f"валидационный якорь d={ANCHOR_VAL} ({DAY0+timedelta(days=ANCHOR_VAL)}), "
      f"таргет-окно {DAY0+timedelta(days=ANCHOR_VAL+1)} .. {DAY0+timedelta(days=ANCHOR_VAL+HORIZON)}")
print(f"боевой якорь        d={ANCHOR_FINAL} ({DAY0+timedelta(days=ANCHOR_FINAL)}), "
      f"таргет-окно {DAY0+timedelta(days=ANCHOR_FINAL+1)} .. {DAY0+timedelta(days=ANCHOR_FINAL+HORIZON)}")

tgt = make_target(ANCHOR_VAL)
y = tgt["y"].to_numpy()
ly = np.log1p(y)
print(f"\nдоля нулей в таргете : {(y==0).mean():.4f}")
print(f"mean(y)              : {y.mean():.3f}   mean(log1p(y)) : {ly.mean():.4f}")
print(f"оптимальная константа: {np.expm1(ly.mean()):.4f}  (наивное среднее было бы {y.mean():.2f})")

валидационный якорь d=378 (2026-01-14), таргет-окно 2026-01-15 .. 2026-02-13
боевой якорь        d=408 (2026-02-13), таргет-окно 2026-02-14 .. 2026-03-15

доля нулей в таргете : 0.4593
mean(y)              : 84.034   mean(log1p(y)) : 2.2421
оптимальная константа: 8.4131  (наивное среднее было бы 84.03)


In [29]:
pos = ly[y > 0]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.09,
                    subplot_titles=["log1p(y) — все пользователи",
                                    "log1p(y) — только покупавшие (y>0)"])
fig.add_trace(hist_trace(ly, bins=80, slot=0), row=1, col=1)
fig.add_trace(hist_trace(pos, bins=80, slot=2), row=1, col=2)
# нормальная аппроксимация положительной части
xs = np.linspace(pos.min(), pos.max(), 200)
pdf = (1 / (pos.std() * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((xs - pos.mean()) / pos.std())**2)
pdf = pdf * len(pos) * (pos.max() - pos.min()) / 80
fig.add_trace(go.Scatter(x=xs, y=pdf, mode="lines", name="норм. приближение",
                         line=dict(color=INK, width=2, dash="dash")), row=1, col=2)
fig.update_yaxes(title="пользователей", row=1, col=1)
fig.update_xaxes(title="log1p(GMV за 30д)", row=1, col=1)
fig.update_xaxes(title="log1p(GMV за 30д)", row=1, col=2)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
fig.update_layout(hovermode="closest", bargap=0.02)
show(fig, "Распределение таргета",
     sub=f"нулей {100*(y==0).mean():.1f}%", h=560)

Картинка — прямое обоснование **hurdle-модели**: смесь «точечная масса в нуле» +
«почти нормальное в `log1p`». Это ровно та структура, ради которой в GA Revenue
делали связку *классификатор ненулевого таргета × регрессия суммы*.

### 4.1 Что на самом деле лежит в `sample_submit.csv`

Файл-пример редко бывает случайным. Проверим его буквально.

In [30]:
sample = pl.read_csv(DATA_START / "sample_submit.csv").sort("user_id")
assert sample["user_id"].to_list() == USERS.to_list(), "состав user_id расходится"
sp = sample["predict"].to_numpy()

print(f"max |sample_submit.predict - y(валидационный якорь)| = {np.abs(sp - y).max():.6f}")
print(f"доля точных совпадений                              = "
      f"{np.mean(np.isclose(sp, y, rtol=1e-6, atol=1e-3)):.4f}")
print(f"RMSLE(sample_submit, y)                             = {rmsle(y, sp):.8f}")

max |sample_submit.predict - y(валидационный якорь)| = 0.001593
доля точных совпадений                              = 1.0000
RMSLE(sample_submit, y)                             = 0.00000002


`sample_submit.csv` — это **не случайная заглушка, а точный факт GMV за предыдущее
30-дневное окно** (2026-01-15 … 2026-02-13), то есть таргет нашего валидационного якоря.

Два практических следствия:

* Отправка `sample_submit.csv` как есть = бейзлайн «прошлые 30 дней» на боевом окне.
  Его качество на валидации мы измеряем строкой *«GMV за прошлые 30д»* ниже — это
  и есть отметка, которую нужно уверенно перебить.
* Он же даёт бесплатный ориентир по распределению: организаторы ожидают на тесте
  примерно такую же долю ненулевых и такой же средний `log1p`. Сильное расхождение
  вашего сабмита с этими числами — повод перепроверить калибровку, а не радоваться.

In [31]:
# --- насколько дорого стоит каждая группа в метрике
grp = np.where(y > 0, "y > 0", "y = 0")
best_const = np.expm1(ly.mean())
pred_const = np.full_like(y, best_const)
err2 = (np.log1p(pred_const) - ly) ** 2
contrib = {g: err2[grp == g].sum() / err2.sum() * 100 for g in ["y = 0", "y > 0"]}
sizes = {g: (grp == g).mean() * 100 for g in ["y = 0", "y > 0"]}
fig = go.Figure()
fig.add_trace(bars(list(sizes), list(sizes.values()), slot=0, name="% пользователей",
                   text=[f"{v:.1f}%" for v in sizes.values()]))
fig.add_trace(bars(list(contrib), list(contrib.values()), slot=1, name="% вклада в MSE",
                   text=[f"{v:.1f}%" for v in contrib.values()]))
fig.update_layout(barmode="group", hovermode="closest")
fig.update_yaxes(title="%")
show(fig, "Кто формирует ошибку константного прогноза", h=520)

In [32]:
# --- линейка простых бейзлайнов на валидационном якоре
def past_sum(anchor_d, days, col="gmv"):
    s = (df.filter(pl.col("d").is_between(anchor_d - days + 1, anchor_d))
           .group_by("user_id").agg(v=pl.col(col).sum().cast(pl.Float64)))
    return (pl.DataFrame({"user_id": USERS}).join(s, on="user_id", how="left")
              .with_columns(pl.col("v").fill_null(0.0))["v"].to_numpy())

g30 = past_sum(ANCHOR_VAL, 30)
g60 = past_sum(ANCHOR_VAL, 60)
g90 = past_sum(ANCHOR_VAL, 90)
g365 = past_sum(ANCHOR_VAL, 365)

baselines = {
    "все нули":                      np.zeros_like(y),
    "среднее y (наивно)":            np.full_like(y, y.mean()),
    "оптимальная константа":         np.full_like(y, best_const),
    "GMV за прошлые 30д":            g30,
    "GMV за прошлые 60д / 2":        g60 / 2,
    "GMV за прошлые 90д / 3":        g90 / 3,
    "GMV за 365д × 30/365":          g365 * 30 / 365,
    "среднее(30д, 90д/3)":           0.5 * (g30 + g90 / 3),
}
# сжатие в log-пространстве: pred = expm1(a * log1p(x)) — подберём a на сетке
grid_a = np.linspace(0.3, 1.2, 19)
sc = [rmsle(y, np.expm1(a * np.log1p(g90 / 3))) for a in grid_a]
a_best = grid_a[int(np.argmin(sc))]
baselines[f"expm1({a_best:.2f}·log1p(90д/3))"] = np.expm1(a_best * np.log1p(g90 / 3))

res = sorted(((k, rmsle(y, v)) for k, v in baselines.items()), key=lambda t: t[1])
for k, v in res:
    print(f"{v:.5f}   {k}")

fig = go.Figure(bars([k for k, _ in res][::-1], [v for _, v in res][::-1], slot=0,
                     orientation="h", text=[f"{v:.4f}" for _, v in res][::-1]))
fig.update_layout(hovermode="closest", xaxis_title="RMSLE (меньше — лучше)")
fig.update_xaxes(range=[0, max(v for _, v in res) * 1.15])
show(fig, "Бейзлайны на валидационном якоре 2026-01-14", h=560, w=980)

1.83614   expm1(0.70·log1p(90д/3))
2.06398   GMV за 365д × 30/365
2.07649   среднее(30д, 90д/3)
2.09355   GMV за прошлые 90д / 3
2.11140   GMV за прошлые 60д / 2
2.19506   GMV за прошлые 30д
2.28829   оптимальная константа
3.17498   среднее y (наивно)
3.20364   все нули


In [33]:
# --- shrinkage-кривая: насколько сильно надо «поджимать» прошлый GMV
alphas = np.linspace(0.0, 1.3, 40)
curves = {}
for nm, base in [("30д", g30), ("90д/3", g90 / 3), ("365д×30/365", g365 * 30 / 365)]:
    curves[nm] = [rmsle(y, np.expm1(a * np.log1p(base))) for a in alphas]
fig = go.Figure()
for i, (nm, c) in enumerate(curves.items()):
    line(fig, alphas, c, nm, slot=i, width=2.5)
    j = int(np.argmin(c))
    fig.add_trace(go.Scatter(x=[alphas[j]], y=[c[j]], mode="markers+text",
                             marker=dict(color=SERIES[i], size=10,
                                         line=dict(color=SURFACE, width=2)),
                             text=[f" a={alphas[j]:.2f}"], textposition="top right",
                             textfont=dict(color=INK2, size=11), showlegend=False))
fig.update_xaxes(title="a  в  pred = expm1(a · log1p(прошлый GMV))")
fig.update_yaxes(title="RMSLE")
fig.update_layout(hovermode="x")
show(fig, "Оптимальное сжатие прошлого GMV в log-пространстве",
     sub="минимум всегда левее единицы", h=560, w=1000)

## 5. Многооконный сэмплинг (схема, которой выиграли GA Revenue)

Победитель Google Analytics Customer Revenue брал **4 непересекающихся окна по 168 дней**,
в каждом считал признаки и таргет, склеивал в один трейн-сет и учил LightGBM.
У нас 409 дней истории — проверим, что помещается, и главное: **сопоставимы ли якоря
между собой**. Если распределение таргета плывёт от якоря к якорю, склейка без поправки
на дрейф испортит калибровку — а по RMSLE калибровка решает.

Ключевое отличие от GA Revenue: там между историей и таргетом был **cooling period 46 дней**,
у нас его нет. Значит recency-признаки будут работать *сильно* лучше, чем в том соревновании,
и переносить их выводы про важность признаков напрямую нельзя.

In [34]:
FEAT_SPAN = 168     # длина окна признаков, как у победителя GA
STRIDE = 30         # шаг между якорями

anchors = []
a = ANCHOR_VAL
while a - FEAT_SPAN + 1 >= 0:
    anchors.append(a)
    a -= STRIDE
anchors = sorted(anchors)
print(f"якорей помещается: {len(anchors)}")
for a in anchors:
    print(f"  d={a:3d}  признаки [{DAY0+timedelta(days=a-FEAT_SPAN+1)} .. {DAY0+timedelta(days=a)}]"
          f"   таргет [{DAY0+timedelta(days=a+1)} .. {DAY0+timedelta(days=a+HORIZON)}]")

# непересекающийся набор в стиле победителя: шаг = FEAT_SPAN + HORIZON
disjoint = []
a = ANCHOR_VAL
while a - FEAT_SPAN + 1 >= 0:
    disjoint.append(a)
    a -= (FEAT_SPAN + HORIZON)
disjoint = sorted(disjoint)
print(f"\nполностью непересекающихся окон по {FEAT_SPAN}д: {len(disjoint)} → {disjoint}")

якорей помещается: 8
  d=168  признаки [2025-01-02 .. 2025-06-18]   таргет [2025-06-19 .. 2025-07-18]
  d=198  признаки [2025-02-01 .. 2025-07-18]   таргет [2025-07-19 .. 2025-08-17]
  d=228  признаки [2025-03-03 .. 2025-08-17]   таргет [2025-08-18 .. 2025-09-16]
  d=258  признаки [2025-04-02 .. 2025-09-16]   таргет [2025-09-17 .. 2025-10-16]
  d=288  признаки [2025-05-02 .. 2025-10-16]   таргет [2025-10-17 .. 2025-11-15]
  d=318  признаки [2025-06-01 .. 2025-11-15]   таргет [2025-11-16 .. 2025-12-15]
  d=348  признаки [2025-07-01 .. 2025-12-15]   таргет [2025-12-16 .. 2026-01-14]
  d=378  признаки [2025-07-31 .. 2026-01-14]   таргет [2026-01-15 .. 2026-02-13]

полностью непересекающихся окон по 168д: 2 → [180, 378]


In [35]:
# --- схема нарезки во времени
fig = go.Figure()
for i, a in enumerate(anchors):
    fs, fe = DAY0 + timedelta(days=a - FEAT_SPAN + 1), DAY0 + timedelta(days=a)
    ts, te = DAY0 + timedelta(days=a + 1), DAY0 + timedelta(days=a + HORIZON)
    fig.add_trace(go.Scatter(x=[fs, fe], y=[i, i], mode="lines", showlegend=(i == 0),
                             name="окно признаков (168д)",
                             line=dict(color=SERIES[0], width=13)))
    fig.add_trace(go.Scatter(x=[ts, te], y=[i, i], mode="lines", showlegend=(i == 0),
                             name="таргет (30д)", line=dict(color=SERIES[1], width=13)))
i = len(anchors)
fig.add_trace(go.Scatter(x=[DAY0 + timedelta(days=ANCHOR_FINAL - FEAT_SPAN + 1),
                            DAY0 + timedelta(days=ANCHOR_FINAL)],
                         y=[i, i], mode="lines", showlegend=False,
                         line=dict(color=SERIES[0], width=13)))
fig.add_trace(go.Scatter(x=[DAY0 + timedelta(days=ANCHOR_FINAL + 1),
                            DAY0 + timedelta(days=ANCHOR_FINAL + HORIZON)],
                         y=[i, i], mode="lines", name="боевой таргет (не виден)",
                         line=dict(color=SERIES[7], width=13, dash="dot")))
fig.add_vline(x=DAY0 + timedelta(days=ANCHOR_FINAL), line=dict(color=INK, width=1.5, dash="dash"))
fig.update_yaxes(tickmode="array", tickvals=list(range(len(anchors) + 1)),
                 ticktext=[(DAY0 + timedelta(days=a)).strftime("%Y-%m-%d") for a in anchors]
                          + [(DAY0 + timedelta(days=ANCHOR_FINAL)).strftime("%Y-%m-%d") + "  (submit)"],
                 title="якорь")
fig.update_layout(hovermode="closest")
show(fig, "Схема многооконной нарезки",
     sub=f"последняя строка — боевой прогноз",
     h=520 + 46 * len(anchors), w=1040)

In [36]:
# --- дрейф таргета между якорями: со всеми пользователями и с фильтром отбора (§3.1)
def drift_row(a, ids=None):
    t = make_target(a)
    if ids is not None:
        t = t.join(pl.DataFrame({"user_id": ids}), on="user_id", how="semi")
    ya = t["y"].to_numpy(); lya = np.log1p(ya)
    return dict(anchor=DAY0 + timedelta(days=a), n=len(ya),
                zero_rate=float((ya == 0).mean()),
                mean_log1p=float(lya.mean()),
                mean_log1p_pos=float(lya[ya > 0].mean()),
                best_const=float(np.expm1(lya.mean())),
                rmsle_const=rmsle(ya, np.full_like(ya, np.expm1(lya.mean()))))

drift = pl.DataFrame([drift_row(a) for a in anchors]).sort("anchor")
drift_sel = pl.DataFrame([drift_row(a, selected_users(a)) for a in anchors]).sort("anchor")
print("все 250K пользователей:")
print(drift)
print("\nтолько прошедшие правило отбора на своём якоре:")
print(drift_sel)

все 250K пользователей:
shape: (8, 7)
┌────────────┬────────┬───────────┬────────────┬────────────────┬────────────┬─────────────┐
│ anchor     ┆ n      ┆ zero_rate ┆ mean_log1p ┆ mean_log1p_pos ┆ best_const ┆ rmsle_const │
│ ---        ┆ ---    ┆ ---       ┆ ---        ┆ ---            ┆ ---        ┆ ---         │
│ date       ┆ i64    ┆ f64       ┆ f64        ┆ f64            ┆ f64        ┆ f64         │
╞════════════╪════════╪═══════════╪════════════╪════════════════╪════════════╪═════════════╡
│ 2025-06-18 ┆ 250000 ┆ 0.551152  ┆ 1.918947   ┆ 4.275272       ┆ 5.813781   ┆ 2.298749    │
│ 2025-07-18 ┆ 250000 ┆ 0.527636  ┆ 2.028156   ┆ 4.293629       ┆ 6.600059   ┆ 2.325361    │
│ 2025-08-17 ┆ 250000 ┆ 0.504032  ┆ 2.137778   ┆ 4.310314       ┆ 7.480571   ┆ 2.347936    │
│ 2025-09-16 ┆ 250000 ┆ 0.488308  ┆ 2.193039   ┆ 4.285857       ┆ 7.962407   ┆ 2.345784    │
│ 2025-10-16 ┆ 250000 ┆ 0.470236  ┆ 2.252792   ┆ 4.252445       ┆ 8.514265   ┆ 2.335952    │
│ 2025-11-15 ┆ 250000 ┆ 0.437316

In [37]:
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.16, horizontal_spacing=0.1,
                    subplot_titles=["Доля нулевого таргета", "mean log1p(y)",
                                    "mean log1p(y) при y>0", "RMSLE оптимальной константы"])
cfg = [("zero_rate", 1, 1), ("mean_log1p", 1, 2),
       ("mean_log1p_pos", 2, 1), ("rmsle_const", 2, 2)]
for c, r, cc in cfg:
    for k, (dd, nm, slot, dash) in enumerate([(drift, "все 250K", 7, "dot"),
                                              (drift_sel, "с фильтром отбора", 0, None)]):
        fig.add_trace(go.Scatter(x=dd["anchor"].to_list(), y=dd[c].to_numpy(),
                                 mode="lines+markers", name=nm,
                                 legendgroup=nm, showlegend=(r == 1 and cc == 1),
                                 line=dict(color=SERIES[slot], width=2.5, dash=dash),
                                 marker=dict(size=8, line=dict(color=SURFACE, width=2))),
                      row=r, col=cc)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2); a.x, a.xanchor = 0, "left"
fig.update_layout(hovermode="x")
show(fig, "Дрейф таргета по якорям",
     h=640, w=1000)

**Как читать этот блок.**

* Расхождение между пунктиром и сплошной линией — это цена игнорирования правила отбора
  (§3.1). Чем дальше якорь в прошлое, тем сильнее «все подряд» перевешены неактивными,
  которых в тесте нет: `zero_rate` завышается на 8–12 пунктов, и обученный на такой
  популяции классификатор систематически недооценивает `P(y>0)` на боевом окне.
* Плоские линии на сплошной кривой ⇒ окна взаимозаменяемы, можно просто склеивать.
  Остаточный тренд лечится: (а) календарными признаками якоря, (б) нормировкой таргета
  на уровень окна, (в) большим весом свежих якорей.
* **Тонкость, которую легко упустить.** На боевом якоре фильтр проходят все 250 000
  по построению. На валидационном якоре `2026-01-14` — только ~93 %. Значит честная
  валидационная популяция это не «все 250K», а `selected_users(378)`: остальные 7 %
  в тестовой популяции просто не встречаются. Ниже — во сколько это обходится в RMSLE.

Отдельно про **дублирование положительных примеров** (в GA Revenue доходило до 60×) —
именно этого делать не надо без возврата prior'а. Если сэмплируете, пересчитывайте
вероятность обратно: `p = p_s / (p_s + (1 - p_s) · w)`, где `w` — коэффициент
передискретизации позитивов. Иначе вы выигрываете AUC и проигрываете RMSLE.

In [38]:
# --- насколько отличается оценка бейзлайна на «всех» и на сопоставимой популяции
sel_val = selected_users(ANCHOR_VAL)
mask_sel = np.isin(USERS.to_numpy(), sel_val.to_numpy())
print(f"валидационная популяция с фильтром: {mask_sel.sum():,} из {N_USERS:,} "
      f"({mask_sel.mean():.1%})\n")
print(f"{'бейзлайн':<32} {'все 250K':>10} {'с фильтром':>12} {'Δ':>8}")
for k, v in baselines.items():
    a_ = rmsle(y, v); b_ = rmsle(y[mask_sel], np.asarray(v)[mask_sel])
    print(f"{k:<32} {a_:>10.4f} {b_:>12.4f} {b_-a_:>+8.4f}")

валидационная популяция с фильтром: 233,152 из 250,000 (93.3%)

бейзлайн                           все 250K   с фильтром        Δ
все нули                             3.2036       3.2756  +0.0720
среднее y (наивно)                   3.1750       3.1142  -0.0608
оптимальная константа                2.2883       2.2966  +0.0083
GMV за прошлые 30д                   2.1951       2.2109  +0.0159
GMV за прошлые 60д / 2               2.1114       2.1217  +0.0103
GMV за прошлые 90д / 3               2.0936       2.1100  +0.0164
GMV за 365д × 30/365                 2.0640       2.0822  +0.0183
среднее(30д, 90д/3)                  2.0765       2.0889  +0.0124
expm1(0.70·log1p(90д/3))             1.8361       1.8488  +0.0126


Сдвиг небольшой по абсолютной величине, но **систематический и в одну сторону** —
а на лидерборде решают третий-четвёртый знак. Практический вывод: держите обе цифры,
но оптимизируйте по популяции с фильтром.

## 6. Построение признаков на якорь

RFM-каркас + оконные агрегации + канальные доли + тренды. Всё считается одним
`group_by` по срезу истории — это дёшево, потому что `d` целочисленный.

In [39]:
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
LAGGED = [(30, 60), (60, 90), (90, 180)]      # окна [a-hi+1 .. a-lo], для трендов


def _w(a, lo, hi):
    return pl.col("d").is_between(a - hi + 1, a - lo)


def build_features(anchor_d: int, hist: int = 365) -> pl.DataFrame:
    """Признаки на момент anchor_d включительно. Без утечки: используются только d <= anchor_d."""
    h = df.filter(pl.col("d").is_between(anchor_d - hist + 1, anchor_d))
    ex = []
    for W in WINDOWS:
        m = _w(anchor_d, 0, W)
        ex += [
            pl.col("gmv").filter(m).sum().alias(f"gmv_{W}"),
            pl.col("to_ord").filter(m).sum().alias(f"ord_{W}"),
            pl.col("to_cart").filter(m).sum().alias(f"cart_{W}"),
            pl.col("searches").filter(m).sum().alias(f"srch_{W}"),
            m.sum().alias(f"act_days_{W}"),
            (m & (pl.col("to_ord") > 0)).sum().alias(f"ord_days_{W}"),
            (m & (pl.col("to_cart") > 0)).sum().alias(f"cart_days_{W}"),
            pl.col("gmv").filter(m & (pl.col("gmv") > 0)).max().alias(f"gmv_max_{W}"),
            pl.col("gmv").filter(m & (pl.col("gmv") > 0)).mean().alias(f"gmv_mean_pos_{W}"),
            pl.col("gmv").filter(m & (pl.col("gmv") > 0)).std().alias(f"gmv_std_pos_{W}"),
        ]
    for lo, hi in LAGGED:
        m = _w(anchor_d, lo, hi)
        ex += [pl.col("gmv").filter(m).sum().alias(f"gmv_lag_{lo}_{hi}"),
               pl.col("to_ord").filter(m).sum().alias(f"ord_lag_{lo}_{hi}"),
               m.sum().alias(f"act_days_lag_{lo}_{hi}")]
    m90 = _w(anchor_d, 0, 90)
    ex += [
        # каналы
        pl.col("gmv_search").filter(m90).sum().alias("gmv_search_90"),
        pl.col("gmv_cat").filter(m90).sum().alias("gmv_cat_90"),
        pl.col("search").filter(m90).sum().alias("search_days_90"),
        pl.col("cat").filter(m90).sum().alias("cat_days_90"),
        pl.col("search_to_ord").filter(m90).sum().alias("s2o_90"),
        pl.col("cat_to_ord").filter(m90).sum().alias("c2o_90"),
        pl.col("search_to_cart").filter(m90).sum().alias("s2c_90"),
        pl.col("cat_to_cart").filter(m90).sum().alias("c2c_90"),
        ((pl.col("search") == 0) & (pl.col("cat") == 0)).filter(m90).sum().alias("phantom_90"),
        (m90 & (pl.col("dow") >= 5)).sum().alias("weekend_days_90"),
        # recency
        (anchor_d - pl.col("d").max()).alias("r_act"),
        (anchor_d - pl.col("d").filter(pl.col("to_ord") > 0).max()).alias("r_ord"),
        (anchor_d - pl.col("d").filter(pl.col("to_cart") > 0).max()).alias("r_cart"),
        (anchor_d - pl.col("d").filter(pl.col("searches") > 0).max()).alias("r_srch"),
        (anchor_d - pl.col("d").filter(pl.col("gmv") > 0).max()).alias("r_gmv"),
        # первое касание внутри истории
        (anchor_d - pl.col("d").min()).alias("hist_span"),
        pl.col("d").filter(pl.col("gmv") > 0).count().alias("n_gmv_days_hist"),
    ]
    f = h.group_by("user_id").agg(ex)

    # tenure считаем по всей истории, а не по срезу
    f = f.join(up.select(["user_id", "first_d"]), on="user_id", how="left") \
         .with_columns(tenure=(anchor_d - pl.col("first_d")).clip(0, None)).drop("first_d")

    eps = 1e-6
    f = f.with_columns(
        aov_90=pl.col("gmv_90") / (pl.col("ord_90") + eps),
        cvr_cart2ord_90=pl.col("ord_90") / (pl.col("cart_90") + eps),
        cvr_srch2cart_90=pl.col("cart_90") / (pl.col("srch_90") + eps),
        buy_rate_90=pl.col("ord_days_90") / (pl.col("act_days_90") + eps),
        cart_rate_90=pl.col("cart_days_90") / (pl.col("act_days_90") + eps),
        density_90=pl.col("act_days_90") / 90.0,
        density_365=pl.col("act_days_365") / 365.0,
        share_search_gmv_90=pl.col("gmv_search_90") / (pl.col("gmv_90") + eps),
        share_search_days_90=pl.col("search_days_90")
            / (pl.col("search_days_90") + pl.col("cat_days_90") + eps),
        phantom_rate_90=pl.col("phantom_90") / (pl.col("act_days_90") + eps),
        weekend_rate_90=pl.col("weekend_days_90") / (pl.col("act_days_90") + eps),
        gmv_per_active_90=pl.col("gmv_90") / (pl.col("act_days_90") + eps),
        srch_per_active_90=pl.col("srch_90") / (pl.col("act_days_90") + eps),
        # тренды: свежее окно против предыдущего, в log-шкале
        trend_gmv_30_60=(pl.col("gmv_30") + 1).log() - (pl.col("gmv_lag_30_60") + 1).log(),
        trend_gmv_60_90=(pl.col("gmv_lag_30_60") + 1).log() - (pl.col("gmv_lag_60_90") + 1).log(),
        trend_act_30_60=(pl.col("act_days_30") + 1).log() - (pl.col("act_days_lag_30_60") + 1).log(),
        gmv_30_over_90=pl.col("gmv_30") / (pl.col("gmv_90") + eps),
        act_30_over_90=pl.col("act_days_30") / (pl.col("act_days_90") + eps),
        # «жив ли» — грубая BTYD-прокси
        alive_proxy=1.0 - (pl.col("r_act").cast(pl.Float64)
                           / (pl.col("hist_span").cast(pl.Float64) / (pl.col("act_days_365") + eps) + eps)
                           ).clip(0, 5) / 5.0,
    )

    # пользователи без истории в окне: заполняем нулями/«очень давно»
    full = pl.DataFrame({"user_id": USERS}).join(f, on="user_id", how="left")
    rec_cols = ["r_act", "r_ord", "r_cart", "r_srch", "r_gmv"]
    full = full.with_columns(
        [pl.col(c).fill_null(999).cast(pl.Int32) for c in rec_cols]
        + [pl.col(c).fill_null(0) for c in full.columns
           if c not in rec_cols + ["user_id"]]
    )
    return full.sort("user_id")


%time F_val = build_features(ANCHOR_VAL)
print(F_val.shape, f"{F_val.estimated_size('mb'):.0f} MB")
F_val.head(3)

CPU times: user 15.7 s, sys: 3.03 s, total: 18.8 s
Wall time: 2.67 s
(250000, 117) 154 MB


user_id,gmv_7,ord_7,cart_7,srch_7,act_days_7,ord_days_7,cart_days_7,gmv_max_7,gmv_mean_pos_7,…,phantom_rate_90,weekend_rate_90,gmv_per_active_90,srch_per_active_90,trend_gmv_30_60,trend_gmv_60_90,trend_act_30_60,gmv_30_over_90,act_30_over_90,alive_proxy
u32,f32,i64,i64,i64,u32,u32,u32,f32,f32,…,f64,f64,f64,f64,f32,f32,f64,f32,f64,f64
2,16.315287,1,2,4,1,1,1,16.315287,16.315287,…,0.0,0.375,2.039411,3.625,2.85159,0.0,0.916291,1.0,0.5,0.950943
7,0.0,0,0,0,0,0,0,0.0,0.0,…,0.096774,0.225806,16.065254,2.612903,0.462715,1.816406,0.143101,0.578855,0.451613,0.60831
15,2242.990967,4,15,21,5,2,3,2237.575684,1121.495483,…,0.333333,0.25,99.911139,1.583333,2.822413,4.902995,0.241162,0.944245,0.541667,0.964345


In [40]:
FEATS = [c for c in F_val.columns if c != "user_id"]
print(f"признаков: {len(FEATS)}")

stat = F_val.select(
    [pl.col(c).eq(0).mean().alias(c) for c in FEATS]
).transpose(include_header=True, header_name="feature", column_names=["zero_share"])
stat = stat.join(
    F_val.select([pl.col(c).cast(pl.Float64).std().alias(c) for c in FEATS])
         .transpose(include_header=True, header_name="feature", column_names=["std"]),
    on="feature")
print("\nпризнаки с нулевой дисперсией или почти всегда нулевые:")
print(stat.filter((pl.col("std") < 1e-9) | (pl.col("zero_share") > 0.98)).sort("zero_share", descending=True))

fig = go.Figure(hist_trace(stat["zero_share"].to_numpy(), bins=40, slot=0))
fig.update_xaxes(title="доля нулей в признаке")
fig.update_yaxes(title="признаков")
fig.update_layout(hovermode="closest", bargap=0.03)
show(fig, "Разреженность самих признаков", h=520)

признаков: 116

признаки с нулевой дисперсией или почти всегда нулевые:
shape: (0, 3)
┌─────────┬────────────┬─────┐
│ feature ┆ zero_share ┆ std │
│ ---     ┆ ---        ┆ --- │
│ str     ┆ f64        ┆ f64 │
╞═════════╪════════════╪═════╡
└─────────┴────────────┴─────┘


## 7. Корреляции

Три среза:
1. связь признака с `log1p(y)` (Спирмен — устойчив к хвостам);
2. связь признака отдельно с **фактом покупки** `1[y>0]` и с **величиной** `log1p(y) | y>0` —
   это заранее показывает, какие признаки работают на «шлагбаум», а какие на «сумму»;
3. взаимные корреляции признаков — что дублируется.

In [41]:
X = F_val.select(FEATS).to_numpy().astype("float32")
yv = tgt["y"].to_numpy()
lyv = np.log1p(yv)
pos_mask = yv > 0


def spearman_vs(Xm, target):
    """Спирмен для матрицы против одного вектора, через ранги."""
    from scipy.stats import rankdata
    tr = rankdata(target)
    tr = (tr - tr.mean()) / (tr.std() + 1e-12)
    out = np.empty(Xm.shape[1])
    for j in range(Xm.shape[1]):
        r = rankdata(Xm[:, j])
        r = (r - r.mean()) / (r.std() + 1e-12)
        out[j] = float(np.mean(r * tr))
    return out


%time corr_all = spearman_vs(X, lyv)
%time corr_bin = spearman_vs(X, pos_mask.astype("float32"))
%time corr_pos = spearman_vs(X[pos_mask], lyv[pos_mask])

corr = (pl.DataFrame({"feature": FEATS, "vs_log1p_y": corr_all,
                      "vs_1[y>0]": corr_bin, "vs_log1p_y_given_pos": corr_pos})
        .with_columns(abs_all=pl.col("vs_log1p_y").abs())
        .sort("abs_all", descending=True))
corr.head(30)

CPU times: user 1.79 s, sys: 184 ms, total: 1.98 s
Wall time: 2.08 s


CPU times: user 1.34 s, sys: 124 ms, total: 1.47 s
Wall time: 1.47 s


CPU times: user 1.01 s, sys: 73.2 ms, total: 1.08 s
Wall time: 1.08 s


feature,vs_log1p_y,vs_1[y>0],vs_log1p_y_given_pos,abs_all
str,f64,f64,f64,f64
"""ord_days_180""",0.643584,0.590949,0.44157,0.643584
"""ord_180""",0.642665,0.583751,0.464983,0.642665
"""ord_365""",0.639056,0.580296,0.462717,0.639056
"""ord_days_90""",0.638088,0.584293,0.436395,0.638088
"""ord_days_365""",0.637933,0.585876,0.436978,0.637933
"""n_gmv_days_hist""",0.637933,0.585876,0.436978,0.637933
"""ord_90""",0.636792,0.577616,0.456899,0.636792
"""s2o_90""",0.631361,0.572126,0.452512,0.631361
"""ord_days_60""",0.6241,0.569169,0.425139,0.6241


In [42]:
topn = corr.head(28).sort("abs_all")
fig = go.Figure()
fig.add_trace(go.Bar(y=topn["feature"], x=topn["vs_log1p_y"], orientation="h",
                     marker=dict(color=[SERIES[0] if v > 0 else SERIES[7]
                                        for v in topn["vs_log1p_y"]],
                                 cornerradius=3, line=dict(color=SURFACE, width=1)),
                     hovertemplate="%{y}: %{x:.3f}<extra></extra>"))
fig.update_xaxes(title="ρ Спирмена с log1p(y)")
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Топ-28 признаков по связи с таргетом",
     sub="красный — обратная связь", h=700, w=980)

In [43]:
# --- какие признаки за «шлагбаум», а какие за «сумму»
fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=corr["vs_1[y>0]"], y=corr["vs_log1p_y_given_pos"], mode="markers",
    marker=dict(size=9, color=SERIES[0], opacity=0.75,
                line=dict(color=SURFACE, width=1.5)),
    text=corr["feature"], hovertemplate="%{text}<br>P(y>0): %{x:.3f}<br>сумма|y>0: %{y:.3f}<extra></extra>"))
lab = corr.filter((pl.col("vs_1[y>0]").abs() > 0.30)
                  | (pl.col("vs_log1p_y_given_pos").abs() > 0.18)).head(14)
fig.add_trace(go.Scattergl(x=lab["vs_1[y>0]"], y=lab["vs_log1p_y_given_pos"],
                           mode="text", text=lab["feature"], textposition="top center",
                           textfont=dict(size=10, color=INK2), showlegend=False))
fig.add_hline(y=0, line=dict(color=AXIS, width=1))
fig.add_vline(x=0, line=dict(color=AXIS, width=1))
fig.update_xaxes(title="ρ с фактом покупки 1[y>0]")
fig.update_yaxes(title="ρ с величиной log1p(y) при y>0")
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Разделение сигнала: «купит ли» против «на сколько»",
     sub="по X — для классификатора, по Y — для регрессии", h=560, w=1000)

Разброс по осям почти всегда асимметричен: связь с *фактом* покупки заметно сильнее,
чем с *величиной*. Это второй аргумент за hurdle: одна модель на всё будет тратить
ёмкость на почти нешумовую задачу «сколько», вместо того чтобы точно калибровать «купит ли».

In [44]:
# --- матрица взаимных корреляций с иерархической кластеризацией
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

SEL = corr.head(45)["feature"].to_list()
Xs = F_val.select(SEL).to_numpy().astype("float64")
Xs = np.sign(Xs) * np.log1p(np.abs(Xs))          # логарифмируем, чтобы Пирсон не съели хвосты
Xs = (Xs - Xs.mean(0)) / (Xs.std(0) + 1e-12)
Cm = np.clip((Xs.T @ Xs) / len(Xs), -1, 1)
np.fill_diagonal(Cm, 1.0)
order = leaves_list(linkage(squareform(np.clip(1 - np.abs(Cm), 0, None), checks=False), "average"))
lab = [SEL[i] for i in order]
fig = go.Figure(heat(Cm[np.ix_(order, order)], lab, lab, colorscale=DIVERGING,
                     zmid=0, zmin=-1, zmax=1, hover="%{z:.2f}"))
fig.update_xaxes(tickangle=-60, tickfont=dict(size=9))
fig.update_yaxes(tickfont=dict(size=9), autorange="reversed")
show(fig, "Корреляции признаков (кластеризованный порядок)",
     sub="тёмные блоки — дублирующиеся семейства", h=780, w=1040)

In [45]:
# --- явный список почти-дублей
pairs = [(lab[i], lab[j], Cm[order[i], order[j]])
         for i in range(len(lab)) for j in range(i + 1, len(lab))
         if abs(Cm[order[i], order[j]]) > 0.95]
print(f"пар с |ρ| > 0.95: {len(pairs)}")
pl.DataFrame(pairs, schema=["feat_a", "feat_b", "rho"], orient="row") \
  .with_columns(pl.col("rho").round(3)).sort("rho", descending=True).head(25)

пар с |ρ| > 0.95: 31


feat_a,feat_b,rho
str,str,f64
"""r_ord""","""r_gmv""",1.0
"""ord_days_365""","""n_gmv_days_hist""",1.0
"""ord_90""","""s2o_90""",0.992
"""gmv_30""","""gmv_max_30""",0.991
"""cart_90""","""s2c_90""",0.988
"""ord_365""","""ord_days_365""",0.987
"""ord_365""","""n_gmv_days_hist""",0.987
"""gmv_60""","""gmv_max_60""",0.986
"""ord_days_180""","""ord_180""",0.984


## 8. Hurdle-разложение: `E[y] = P(y>0) · E[y | y>0]`

Здесь смотрим, чем именно предсказывается каждая часть, и заодно получаем
готовые «карты», по которым можно проверять калибровку модели.

In [46]:
def bin_profile(x, y_true, edges, labels=None):
    idx = np.digitize(x, edges[1:-1])
    rows = []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() < 50:
            continue
        yy = y_true[m]
        rows.append(dict(bin=(labels[b] if labels else f"[{edges[b]:g},{edges[b+1]:g})"),
                         n=int(m.sum()), p_pos=float((yy > 0).mean()),
                         mean_log1p_pos=float(np.log1p(yy[yy > 0]).mean()) if (yy > 0).any() else 0.0,
                         mean_log1p=float(np.log1p(yy).mean())))
    return pl.DataFrame(rows)


r_edges = np.array([0, 1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60, 90, 150, 250, 1000])
prof_r = bin_profile(F_val["r_act"].to_numpy(), yv, r_edges)

fig = make_subplots(specs=[[{"secondary_y": False}]])
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=["P(y > 0)", "E[log1p(y) | y > 0]"])
fig.add_trace(go.Scatter(x=prof_r["bin"], y=prof_r["p_pos"], mode="lines+markers",
                         line=dict(color=SERIES[0], width=2.5),
                         marker=dict(size=9, line=dict(color=SURFACE, width=2)),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=prof_r["bin"], y=prof_r["mean_log1p_pos"], mode="lines+markers",
                         line=dict(color=SERIES[1], width=2.5),
                         marker=dict(size=9, line=dict(color=SURFACE, width=2)),
                         showlegend=False), row=1, col=2)
fig.update_xaxes(title="дней с последней активности", row=1, col=1)
fig.update_xaxes(title="дней с последней активности", row=1, col=2)
fig.update_yaxes(rangemode="tozero", row=1, col=1)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
fig.update_layout(hovermode="x")
show(fig, "Recency против обеих частей hurdle",
     sub="recency объясняет «шлагбаум», но не сумму", h=560)
prof_r

bin,n,p_pos,mean_log1p_pos,mean_log1p
str,i64,f64,f64,f64
"""[0,1)""",92245,0.70367,4.332328,3.048528
"""[1,2)""",37833,0.599609,4.143086,2.484231
"""[2,3)""",23962,0.532259,4.033997,2.147133
"""[3,5)""",30292,0.46537,3.940047,1.833581
"""[5,7)""",16364,0.392813,3.847627,1.5114
"""[7,10)""",14567,0.346605,3.774262,1.308179
"""[10,14)""",11095,0.30311,3.733092,1.131536
"""[14,21)""",13318,0.280598,3.753809,1.05331
"""[21,30)""",10324,0.207478,3.717921,0.771386


In [47]:
# --- 2D карта: recency × частота
rb = np.array([0, 1, 3, 7, 14, 30, 60, 120, 1000])
fb = np.array([0, 1, 2, 4, 8, 16, 32, 64, 400])
ri = np.digitize(F_val["r_act"].to_numpy(), rb[1:-1])
fi = np.digitize(F_val["ord_days_90"].to_numpy(), fb[1:-1])
Pp = np.full((len(fb) - 1, len(rb) - 1), np.nan)
Ml = np.full_like(Pp, np.nan)
Nn = np.zeros_like(Pp)
for i in range(len(fb) - 1):
    for j in range(len(rb) - 1):
        m = (fi == i) & (ri == j)
        Nn[i, j] = m.sum()
        if m.sum() >= 50:
            yy = yv[m]
            Pp[i, j] = (yy > 0).mean()
            if (yy > 0).any():
                Ml[i, j] = np.log1p(yy[yy > 0]).mean()
rl = [f"{rb[j]}–{rb[j+1]}" for j in range(len(rb) - 1)]
fl = [f"{fb[i]}–{fb[i+1]}" for i in range(len(fb) - 1)]

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.075,
                    subplot_titles=["P(y > 0)", "E[log1p(y) | y > 0]", "пользователей (log10)"])
fig.add_trace(heat(Pp, rl, fl, hover="%{z:.3f}"), row=1, col=1)
fig.add_trace(heat(Ml, rl, fl, hover="%{z:.2f}"), row=1, col=2)
fig.add_trace(heat(np.log10(Nn + 1), rl, fl, hover="%{z:.2f}"), row=1, col=3)
for k, t in enumerate(fig.data):
    t.colorbar = dict(thickness=10, outlinewidth=0, len=0.85, x=0.29 + 0.354 * k,
                      tickfont=dict(color=MUTED, size=10))
for i in range(1, 4):
    fig.update_xaxes(title="recency, дней" if i == 2 else None, row=1, col=i)
fig.update_yaxes(title="дней с покупкой за 90д", row=1, col=1)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
show(fig, "RF-карта: recency × frequency",
     sub="правая панель — контроль плотности данных", h=560, w=1060)

In [48]:
# --- калибровка: децили по прошлому GMV
dec = np.clip(np.searchsorted(np.quantile(g90[g90 > 0], np.linspace(0, 1, 11))[1:-1], g90), 0, 9)
dec = np.where(g90 == 0, -1, dec)
rows = []
for dv in sorted(set(dec.tolist())):
    m = dec == dv
    rows.append(dict(дециль=("нет GMV за 90д" if dv < 0 else f"D{dv+1}"),
                     n=int(m.sum()),
                     прошлый_GMV_ср=float(g90[m].mean() / 3),
                     факт_y_ср=float(yv[m].mean()),
                     P_y_pos=float((yv[m] > 0).mean()),
                     mean_log1p_y=float(np.log1p(yv[m]).mean())))
calib = pl.DataFrame(rows)
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=["Прошлое против будущего (шкала log1p)",
                                    "Доля покупающих по децилям"])
fig.add_trace(go.Bar(x=calib["дециль"], y=np.log1p(calib["прошлый_GMV_ср"]), name="log1p прошлый GMV/мес",
                     marker=dict(color=SERIES[0], cornerradius=3, line=dict(color=SURFACE, width=1))),
              row=1, col=1)
fig.add_trace(go.Bar(x=calib["дециль"], y=np.log1p(calib["факт_y_ср"]), name="log1p факт y",
                     marker=dict(color=SERIES[1], cornerradius=3, line=dict(color=SURFACE, width=1))),
              row=1, col=1)
fig.add_trace(go.Scatter(x=calib["дециль"], y=calib["P_y_pos"], mode="lines+markers",
                         line=dict(color=SERIES[2], width=2.5), showlegend=False,
                         marker=dict(size=9, line=dict(color=SURFACE, width=2))), row=1, col=2)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
fig.update_layout(barmode="group", hovermode="x")
show(fig, "Регрессия к среднему по децилям прошлого GMV", h=560)
calib

дециль,n,прошлый_GMV_ср,факт_y_ср,P_y_pos,mean_log1p_y
str,i64,f64,f64,f64,f64
"""нет GMV за 90д""",62512,0.0,10.818085,0.156802,0.535353
"""D1""",18749,3.62247,20.29564,0.365299,1.193094
"""D2""",18749,9.691217,27.141079,0.442904,1.503658
"""D3""",18749,18.065895,34.799551,0.523388,1.846672
"""D4""",18748,29.252009,44.286771,0.587049,2.146039
"""D5""",18749,44.514024,57.424095,0.661635,2.524026
"""D6""",18749,65.834984,74.116725,0.719398,2.863209
"""D7""",18748,96.73212,97.937705,0.776296,3.262309
"""D8""",18749,145.366597,126.631942,0.82111,3.635222


## 9. Фурье и спектральный анализ

Два уровня:
* **глобальный** — периодограмма и ACF дневных рядов площадки: подтвердить недельный цикл
  и найти длинные периодичности;
* **пользовательский** — есть ли у отдельного человека устойчивый ритм, и переносится ли
  он в полезный признак. Проверяем прямо: считаем DFT последних 168 дней активности
  и меряем корреляцию гармоник с таргетом.

In [49]:
def periodogram(x, detrend_win=28):
    x = np.asarray(x, dtype="float64")
    k = np.ones(detrend_win) / detrend_win
    trend = np.convolve(x, k, mode="same")
    r = (x - trend)[detrend_win: -detrend_win]
    r = r - r.mean()
    sp = np.abs(np.fft.rfft(r * np.hanning(len(r))))**2
    fr = np.fft.rfftfreq(len(r), d=1.0)
    return fr[1:], sp[1:]


fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.09,
                    subplot_titles=["Периодограмма (детрендировано MA-28)",
                                    "Автокорреляция остатков"])
for i, (col, nm) in enumerate([("dau", "DAU"), ("gmv", "GMV"), ("searches", "запросы")]):
    fr, sp = periodogram(daily[col].to_numpy())
    per = 1 / fr
    m = (per >= 2) & (per <= 120)
    fig.add_trace(go.Scatter(x=per[m], y=sp[m] / sp[m].max(), mode="lines", name=nm,
                             line=dict(color=SERIES[i], width=2)), row=1, col=1)
for p_ in [7, 3.5, 2.333]:
    fig.add_vline(x=p_, line=dict(color=AXIS, width=1, dash="dot"), row=1, col=1)

def acf(x, nlag=70, detrend_win=28):
    x = np.asarray(x, "float64")
    k = np.ones(detrend_win) / detrend_win
    r = (x - np.convolve(x, k, mode="same"))[detrend_win:-detrend_win]
    r = r - r.mean()
    d = np.dot(r, r)
    return np.array([np.dot(r[:len(r)-l], r[l:]) / d for l in range(nlag + 1)])

for i, (col, nm) in enumerate([("dau", "DAU"), ("gmv", "GMV"), ("searches", "запросы")]):
    a_ = acf(daily[col].to_numpy())
    fig.add_trace(go.Scatter(x=np.arange(len(a_)), y=a_, mode="lines", name=nm,
                             line=dict(color=SERIES[i], width=2), showlegend=False), row=1, col=2)
fig.add_hline(y=0, line=dict(color=AXIS, width=1), row=1, col=2)
fig.update_xaxes(title="период, дней", type="log", row=1, col=1)
fig.update_xaxes(title="лаг, дней", row=1, col=2)
fig.update_yaxes(title="нормированная мощность", row=1, col=1)
fig.update_yaxes(title="ACF", row=1, col=2)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
fig.update_layout(hovermode="x")
show(fig, "Спектр глобальных дневных рядов",
     sub="пунктиры — 7 дней и гармоники 7/2, 7/3", h=560, w=1000)

In [50]:
# --- пользовательский спектр на плотной матрице (только выборка)
SPEC_N = min(SAMPLE_USERS, N_USERS)
SPEC_W = 168
spec_ids = np.sort(RNG.choice(USERS.to_numpy(), size=SPEC_N, replace=False))
srow = {u: i for i, u in enumerate(spec_ids)}
s = df.filter(pl.col("user_id").is_in(spec_ids)
              & pl.col("d").is_between(ANCHOR_VAL - SPEC_W + 1, ANCHOR_VAL)) \
      .select(["user_id", "d", "gmv"])
A = np.zeros((SPEC_N, SPEC_W), dtype="float32")
G = np.zeros_like(A)
ii = np.array([srow[u] for u in s["user_id"].to_numpy()])
jj = (s["d"].to_numpy() - (ANCHOR_VAL - SPEC_W + 1)).astype(int)
A[ii, jj] = 1.0
G[ii, jj] = s["gmv"].to_numpy()
print(f"матрица активности {A.shape}, заполненность {A.mean():.3f}, "
      f"{A.nbytes/2**20:.0f} MB")

Ac = A - A.mean(1, keepdims=True)
S = np.abs(np.fft.rfft(Ac, axis=1))                # SPEC_N × 85
freqs = np.fft.rfftfreq(SPEC_W, d=1.0)
periods = np.divide(1, freqs, out=np.full_like(freqs, np.inf), where=freqs > 0)

mean_spec = S.mean(0)
m = (periods >= 2) & (periods <= 90)
fig = go.Figure()
line(fig, periods[m], mean_spec[m] / mean_spec[m].max(), "средняя мощность по пользователям",
     slot=0, width=2.5)
for p_ in [7, 3.5, 2.333]:
    fig.add_vline(x=p_, line=dict(color=AXIS, width=1, dash="dot"))
fig.update_xaxes(title="период, дней", type="log")
fig.update_yaxes(title="нормированная |DFT|")
show(fig, f"Усреднённый спектр индивидуальной активности ({SPEC_N:,} пользователей, 168 дней)",
     sub="пик на 7 днях — недельный ритм", h=560)

матрица активности (60000, 168), заполненность 0.339, 38 MB


In [51]:
# --- насколько выражен недельный ритм у конкретного человека
k7 = int(np.argmin(np.abs(periods - 7.0)))
pw = S[:, 1:]                                       # без DC
tot = pw.sum(1) + 1e-9
w7 = (S[:, k7] + S[:, max(k7-1, 1)] + S[:, min(k7+1, S.shape[1]-1)]) / tot
dom_p = periods[1:][np.argmax(pw, axis=1)]
act_cnt = A.sum(1)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.09,
                    subplot_titles=["Доля мощности в недельной полосе",
                                    "Доминирующий период (среди активных ≥ 20 дней)"])
fig.add_trace(hist_trace(w7[act_cnt >= 10], bins=60, slot=0), row=1, col=1)
dp = dom_p[act_cnt >= 20]
dp = dp[np.isfinite(dp) & (dp <= 90)]
fig.add_trace(hist_trace(dp, bins=60, slot=1), row=1, col=2)
fig.update_xaxes(title="power(≈7д) / total power", row=1, col=1)
fig.update_xaxes(title="период, дней", row=1, col=2)
fig.update_yaxes(title="пользователей", row=1, col=1)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
fig.update_layout(showlegend=False, hovermode="closest", bargap=0.03)
show(fig, "Индивидуальная периодичность", h=560)
print(f"медианная доля недельной мощности: {np.median(w7[act_cnt>=10]):.3f}")

медианная доля недельной мощности: 0.036


In [52]:
# --- проверка на полезность: коррелируют ли гармоники с таргетом
y_spec = tgt.filter(pl.col("user_id").is_in(spec_ids)).sort("user_id")["y"].to_numpy()
ly_spec = np.log1p(y_spec)
K = 20
harm = S[:, 1:K + 1] / (S[:, :1] + 1e-6)            # нормируем на DC = общий уровень активности
from scipy.stats import rankdata
def sp1(a, b):
    ra = rankdata(a); rb = rankdata(b)
    ra = (ra - ra.mean()) / (ra.std() + 1e-12); rb = (rb - rb.mean()) / (rb.std() + 1e-12)
    return float(np.mean(ra * rb))

rho_h = [sp1(harm[:, k], ly_spec) for k in range(K)]
rho_ref = {"DC (уровень активности)": sp1(S[:, 0], ly_spec),
           "число активных дней":     sp1(act_cnt, ly_spec),
           "доля недельной мощности":  sp1(w7, ly_spec)}
fig = go.Figure()
fig.add_trace(bars([f"{periods[k+1]:.1f}д" for k in range(K)], rho_h, slot=0,
                   name="нормированные гармоники"))
for i, (nm, v) in enumerate(rho_ref.items()):
    fig.add_hline(y=v, line=dict(color=SERIES[(i + 1) % 8], width=2, dash="dash"),
                  annotation_text=f"{nm}: {v:.3f}", annotation_position="top left",
                  annotation_font=dict(color=SERIES[(i + 1) % 8], size=11))
fig.update_xaxes(title="период гармоники")
fig.update_yaxes(title="ρ Спирмена с log1p(y)")
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Полезны ли Фурье-признаки сами по себе",
     sub="пунктиры — простые альтернативы", h=560, w=1000)

**Как это читать.** Мощность DC — это просто уровень активности, он и должен доминировать.
Смысл имеют *нормированные* гармоники: они кодируют не «сколько», а «как ритмично».
Если их корреляция ощутимо ниже базовых признаков (что типично), Фурье идёт как
небольшой дополнительный блок, а не как основа. Заметно дешевле и обычно полезнее —
профиль по дням недели (7 чисел) плюс `weekend_rate`, которые мы уже посчитали в §6.

## 10. Токенизация дневного поведения

ТЗ прямо намекает на seq-модели. Минимальный словарь: день описывается битами
`(search, cat, cart>0, ord>0)` → 16 состояний, плюс отдельный токен `ABSENT`
для дня без строки. Отсюда видно и частоты состояний, и матрицу переходов —
основу для любой авторегрессионной модели.

In [53]:
TOK_W = 120
tok_ids = spec_ids[:20_000]
trow = {u: i for i, u in enumerate(tok_ids)}
t = df.filter(pl.col("user_id").is_in(tok_ids)
              & pl.col("d").is_between(ANCHOR_VAL - TOK_W + 1, ANCHOR_VAL)) \
      .select(["user_id", "d", "search", "cat", "to_cart", "to_ord"])
T = np.zeros((len(tok_ids), TOK_W), dtype="int8")            # 0 = ABSENT
ii = np.array([trow[u] for u in t["user_id"].to_numpy()])
jj = (t["d"].to_numpy() - (ANCHOR_VAL - TOK_W + 1)).astype(int)
code_ = (t["search"].to_numpy().astype("int8") * 1
         + t["cat"].to_numpy().astype("int8") * 2
         + (t["to_cart"].to_numpy() > 0).astype("int8") * 4
         + (t["to_ord"].to_numpy() > 0).astype("int8") * 8)
T[ii, jj] = code_ + 1                                        # 1..16

def tok_name(k):
    if k == 0:
        return "ABSENT"
    b = k - 1
    p = []
    if b & 1: p.append("S")
    if b & 2: p.append("C")
    if b & 4: p.append("cart")
    if b & 8: p.append("ord")
    return "·".join(p) if p else "—"

names = [tok_name(k) for k in range(17)]
cnt = np.bincount(T.ravel(), minlength=17)
o = np.argsort(-cnt)
fig = go.Figure(bars([names[k] for k in o], cnt[o] / cnt.sum() * 100, slot=0,
                     text=[f"{v:.2f}%" for v in cnt[o] / cnt.sum() * 100]))
fig.update_yaxes(title="% всех дней", type="log")
fig.update_layout(hovermode="closest")
show(fig, "Частоты дневных токенов",
     sub="S — Поиск, C — Каталог; ось Y логарифмическая", h=560, w=1000)

In [54]:
# --- матрица переходов день→день
nxt = T[:, 1:].ravel(); cur = T[:, :-1].ravel()
M = np.zeros((17, 17))
np.add.at(M, (cur, nxt), 1)
keep = [k for k in range(17) if cnt[k] / cnt.sum() > 0.0008]
P = M[np.ix_(keep, keep)]
P = P / (P.sum(1, keepdims=True) + 1e-9)
lbl = [names[k] for k in keep]
fig = go.Figure(heat(np.log10(P + 1e-4), lbl, lbl, hover="log10 P = %{z:.2f}"))
fig.update_xaxes(title="состояние завтра", tickangle=-45)
fig.update_yaxes(title="состояние сегодня", autorange="reversed")
show(fig, "Матрица переходов между дневными состояниями (log10 вероятности)",
     sub="столбец ABSENT — та же разреженность", h=560, w=980)

In [55]:
# --- энтропия поведения пользователя vs таргет
Pu = np.zeros((len(tok_ids), 17))
for k in range(17):
    Pu[:, k] = (T == k).sum(1)
Pu = Pu / Pu.sum(1, keepdims=True)
ent = -(Pu * np.log(Pu + 1e-12)).sum(1)
y_tok = tgt.filter(pl.col("user_id").is_in(tok_ids)).sort("user_id")["y"].to_numpy()
print(f"ρ(энтропия поведения, log1p(y)) = {sp1(ent, np.log1p(y_tok)):.3f}")

qs = np.quantile(ent, np.linspace(0, 1, 11))
bi = np.clip(np.digitize(ent, qs[1:-1]), 0, 9)
pp = [float((y_tok[bi == b] > 0).mean()) for b in range(10)]
ml = [float(np.log1p(y_tok[bi == b]).mean()) for b in range(10)]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=["P(y>0) по децилям энтропии", "mean log1p(y)"])
fig.add_trace(bars([f"D{i+1}" for i in range(10)], pp, slot=0), row=1, col=1)
fig.add_trace(bars([f"D{i+1}" for i in range(10)], ml, slot=1), row=1, col=2)
for a in fig.layout.annotations:
    a.font = dict(size=12, color=INK2)
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Разнообразие поведения как признак", h=520)

ρ(энтропия поведения, log1p(y)) = 0.511


## 11. Causal inference

Соревнование чисто предсказательное, поэтому причинность здесь нужна не ради лидерборда,
а ради двух вещей: (1) понять, какие сигналы *механически* ведут к GMV, а какие
просто маркируют активного человека; (2) не построить модель на признаке, который
на самом деле — следствие таргета или общего конфаундера.

Готовых `dowhy`/`econml` в окружении нет, но всё нужное здесь короткое, поэтому
оценки реализованы напрямую: **event-study**, **propensity score + IPW/матчинг**
с проверкой баланса ковариат, и **Granger-стиль тест предсказательной причинности**
с фиксированными эффектами пользователя.

> Все оценки — под допущением *selection on observables*. Рандомизации нет, это не A/B;
> корректная интерпретация — «разница между сопоставимыми по наблюдаемым признакам группами».

### 11.1 Event study вокруг промо

Промо-день — квазиэксперимент: резкий шок спроса. Интересен не сам всплеск, а
**что происходит после**: если после пика идёт провал ниже базовой линии, значит
промо *переносит* спрос, а не создаёт — и `gmv_30d`, захвативший промо, систематически
завышает прогноз на следующие 30 дней.

In [56]:
peaks = daily.sort("gmv", descending=True).head(40)
sel, used = [], []
for r in peaks.iter_rows(named=True):
    if all(abs(r["d"] - u) > 21 for u in used):
        sel.append(r); used.append(r["d"])
    if len(sel) == 4:
        break
sel = sorted(sel, key=lambda r: r["d"])

W = 21
base_lvl = daily["gmv"].median()
fig = go.Figure()
for i, r in enumerate(sel):
    a = r["d"]
    lo, hi = max(0, a - W), min(N_DAYS - 1, a + W)
    seg = daily.filter(pl.col("d").is_between(lo, hi)).sort("d")
    # базовая линия = медиана того же дня недели за 8 недель ДО события
    pre = daily.filter(pl.col("d").is_between(a - 56, a - 8))
    bl = pre.group_by("dow").agg(b=pl.col("gmv").median())
    seg = seg.join(bl, on="dow", how="left")
    rel = (seg["gmv"] / seg["b"]).to_numpy()
    line(fig, (seg["d"] - a).to_numpy(), rel,
         (DAY0 + timedelta(days=a)).strftime("%Y-%m-%d"), slot=i, width=2.2)
fig.add_hline(y=1.0, line=dict(color=AXIS, width=1.5, dash="dash"))
fig.add_vline(x=0, line=dict(color=INK, width=1.5))
fig.update_xaxes(title="дней от пикового дня")
fig.update_yaxes(title="GMV / базовая линия того же дня недели", type="log")
fig.update_layout(hovermode="x")
show(fig, "Event study: GMV вокруг пиковых дней",
     sub="1.0 — обычный уровень того же дня недели", h=560)

for r in sel:
    a = r["d"]
    pre = daily.filter(pl.col("d").is_between(a - 14, a - 1))["gmv"].sum()
    post = daily.filter(pl.col("d").is_between(a + 1, a + 14))["gmv"].sum()
    print(f"{(DAY0+timedelta(days=a)).isoformat()}: пик ×{r['gmv']/base_lvl:.2f} к медиане | "
          f"14д после / 14д до = {post/pre:.3f}")

2025-10-15: пик ×1.32 к медиане | 14д после / 14д до = 0.985
2025-11-11: пик ×1.42 к медиане | 14д после / 14д до = 1.116
2025-12-03: пик ×1.32 к медиане | 14д после / 14д до = 1.142
2025-12-27: пик ×1.65 к медиане | 14д после / 14д до = 0.715


### 11.2 Propensity score: интенсивность Поиска против Каталога

Наивное сравнение GMV у «поисковых» и «каталожных» пользователей смешано с конфаундером
«общая активность»: кто больше делает всего, тот больше и ищет, и покупает.

Важная деталь, которая всплыла при первой попытке: **чистых «каталожных» пользователей
в данных практически нет** — медианная доля поисковых дней ≈ 0.90, и порог вида
«≥0.75 против ≤0.25» даёт 99.7 % против 0.3 %, то есть контрольной группы нет вовсе,
а любая оценка превращается в экстраполяцию. Поэтому лечение определяем **квантильно**:
верхний квартиль по доле Поиска против нижнего. Это честный контраст интенсивности
внутри реально существующей популяции.

Сравниваем три оценки: наивную разницу, IPW и матчинг 1:1 по скору.
Главная диагностика — **love plot** стандартизованных разностей до и после взвешивания.

In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

CONF = ["tenure", "r_act", "act_days_90", "act_days_365", "ord_days_90",
        "gmv_lag_30_60", "gmv_lag_60_90", "srch_per_active_90",
        "cart_90", "density_365", "aov_90"]

# лечение определяем по ПРОШЛОМУ поведению (до якоря) — иначе утечка
base = (F_val.with_columns(share=pl.col("share_search_days_90"))
             .filter((pl.col("act_days_90") >= 5) & (pl.col("gmv_90") > 0)))
q_lo, q_hi = np.quantile(base["share"].to_numpy(), [0.25, 0.75])
print(f"доля поисковых дней: p25 = {q_lo:.3f}, p75 = {q_hi:.3f} "
      f"(медиана {np.median(base['share'].to_numpy()):.3f}) — "
      "чистого «каталожного» сегмента в данных нет, поэтому режем по квартилям")
base = base.with_columns(
    W=pl.when(pl.col("share") >= q_hi).then(1)
      .when(pl.col("share") <= q_lo).then(0).otherwise(None)
).drop_nulls("W")

yb = tgt.join(base.select("user_id"), on="user_id", how="semi").sort("user_id")["y"].to_numpy()
base = base.sort("user_id")
Wt = base["W"].to_numpy().astype(int)
Xc = base.select(CONF).to_numpy().astype("float64")
Xc = np.sign(Xc) * np.log1p(np.abs(Xc))
Xc = StandardScaler().fit_transform(Xc)
lyb = np.log1p(yb)
print(f"выборка {len(Wt):,}   верхний квартиль по Поиску (treated) {Wt.sum():,} ({Wt.mean():.1%})")

ps = LogisticRegression(max_iter=1000, C=1.0).fit(Xc, Wt).predict_proba(Xc)[:, 1]
print(f"propensity: [{ps.min():.3f}, {ps.max():.3f}], медиана {np.median(ps):.3f}")

fig = go.Figure()
for i, (g, nm) in enumerate([(1, "верхний квартиль по Поиску (treated)"),
                             (0, "нижний квартиль по Поиску (control)")]):
    c, e = np.histogram(ps[Wt == g], bins=50, range=(0, 1), density=True)
    fig.add_trace(go.Bar(x=(e[:-1] + e[1:]) / 2, y=c, name=nm, opacity=0.65,
                         marker=dict(color=SERIES[i], line=dict(color=SURFACE, width=0.4))))
fig.update_layout(barmode="overlay", hovermode="x", bargap=0.02)
fig.update_xaxes(title="propensity score")
fig.update_yaxes(title="плотность")
show(fig, "Общая область определения (common support)", h=520)

доля поисковых дней: p25 = 0.800, p75 = 0.962 (медиана 0.899) — чистого «каталожного» сегмента в данных нет, поэтому режем по квартилям
выборка 93,468   верхний квартиль по Поиску (treated) 46,755 (50.0%)
propensity: [0.014, 0.985], медиана 0.508


In [58]:
# --- IPW (стабилизированные веса) + матчинг 1:1
eps = 1e-3
psc = np.clip(ps, eps, 1 - eps)
w_att = np.where(Wt == 1, 1.0, psc / (1 - psc))          # веса под ATT
w_att = w_att / np.array([w_att[Wt == g].mean() for g in Wt])

naive = lyb[Wt == 1].mean() - lyb[Wt == 0].mean()
ipw = (np.average(lyb[Wt == 1], weights=w_att[Wt == 1])
       - np.average(lyb[Wt == 0], weights=w_att[Wt == 0]))

tr, ct = np.where(Wt == 1)[0], np.where(Wt == 0)[0]
nn = NearestNeighbors(n_neighbors=1).fit(np.log(psc[ct] / (1 - psc[ct])).reshape(-1, 1))
dist, ind = nn.kneighbors(np.log(psc[tr] / (1 - psc[tr])).reshape(-1, 1))
ok = dist.ravel() < 0.05                                  # caliper
matched = lyb[tr[ok]] - lyb[ct[ind.ravel()[ok]]]
match_est = matched.mean()
bs = np.array([matched[RNG.integers(0, len(matched), len(matched))].mean() for _ in range(300)])
ci = np.percentile(bs, [2.5, 97.5])
print(f"наивная разница : {naive:+.4f}")
print(f"IPW (ATT)       : {ipw:+.4f}")
print(f"матчинг 1:1     : {match_est:+.4f}   95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]   "
      f"пар {ok.sum():,} ({ok.mean():.1%} treated)")

fig = go.Figure()
est = [("наивная разница", naive, None), ("IPW (ATT)", ipw, None),
       ("матчинг 1:1", match_est, ci)]
fig.add_trace(go.Bar(x=[e[0] for e in est], y=[e[1] for e in est],
                     marker=dict(color=[SERIES[7], SERIES[0], SERIES[2]], cornerradius=4,
                                 line=dict(color=SURFACE, width=1)),
                     error_y=dict(type="data", symmetric=False,
                                  array=[0, 0, ci[1] - match_est],
                                  arrayminus=[0, 0, match_est - ci[0]],
                                  color=INK2, thickness=1.5, width=6),
                     text=[f"{e[1]:+.3f}" for e in est], textposition="outside",
                     textfont=dict(color=INK2, size=12), cliponaxis=False))
fig.add_hline(y=0, line=dict(color=AXIS, width=1.5))
fig.update_yaxes(title="Δ mean log1p(GMV за 30д)")
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Эффект высокой интенсивности Поиска на GMV следующих 30 дней",
     sub="до и после поправки на активность", h=560)

наивная разница : -0.2549
IPW (ATT)       : +0.0518
матчинг 1:1     : +0.0224   95% CI [-0.0021, +0.0491]   пар 46,751 (100.0% treated)


In [59]:
# --- love plot: баланс ковариат до и после взвешивания
def smd(x, w=None):
    if w is None:
        w = np.ones(len(x))
    m1 = np.average(x[Wt == 1], weights=w[Wt == 1])
    m0 = np.average(x[Wt == 0], weights=w[Wt == 0])
    v1 = np.average((x[Wt == 1] - m1)**2, weights=w[Wt == 1])
    v0 = np.average((x[Wt == 0] - m0)**2, weights=w[Wt == 0])
    return (m1 - m0) / np.sqrt((v1 + v0) / 2 + 1e-12)

before = np.array([smd(Xc[:, j]) for j in range(Xc.shape[1])])
after = np.array([smd(Xc[:, j], w_att) for j in range(Xc.shape[1])])
o = np.argsort(np.abs(before))
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.abs(before)[o], y=[CONF[i] for i in o], mode="markers",
                         name="до взвешивания",
                         marker=dict(color=SERIES[7], size=11, line=dict(color=SURFACE, width=1.5))))
fig.add_trace(go.Scatter(x=np.abs(after)[o], y=[CONF[i] for i in o], mode="markers",
                         name="после IPW",
                         marker=dict(color=SERIES[0], size=11, line=dict(color=SURFACE, width=1.5))))
fig.add_vline(x=0.1, line=dict(color=STATUS["warning"], width=1.5, dash="dash"),
              annotation_text="порог 0.1", annotation_font=dict(color=INK2, size=11))
fig.update_xaxes(title="|стандартизованная разность средних|")
fig.update_layout(hovermode="closest")
show(fig, "Love plot: баланс ковариат",
     sub="правее 0.1 — остаточный дисбаланс", h=560, w=980)

### 11.3 Granger-стиль: предсказательная причинность внутри пользователя

Вопрос: несёт ли поведение (корзины, запросы) информацию о будущем GMV **сверх** того,
что уже сказано прошлым GMV? Панель по неделям, фиксированные эффекты пользователя
(вычитаем средние по пользователю — так убираем всю межпользовательскую вариацию,
т.е. главный конфаундер «человек в принципе активный»), затем сравниваем `R²`
вложенных регрессий.

In [60]:
GR_N = 30_000
gids = np.sort(RNG.choice(USERS.to_numpy(), size=GR_N, replace=False))
wk = (df.filter(pl.col("user_id").is_in(gids))
        .with_columns(w=(pl.col("d") // 7).cast(pl.Int16))
        .group_by(["user_id", "w"]).agg(
            gmv=pl.col("gmv").sum(), cart=pl.col("to_cart").sum(),
            srch=pl.col("searches").sum(), act=pl.len()))
n_w = int(df["d"].max() // 7) + 1
rowi = {u: i for i, u in enumerate(gids)}
ri_ = np.array([rowi[u] for u in wk["user_id"].to_numpy()])
ci_ = wk["w"].to_numpy().astype(int)
P_ = {}
for nm in ["gmv", "cart", "srch", "act"]:
    Mx = np.zeros((GR_N, n_w), dtype="float32")
    Mx[ri_, ci_] = wk[nm].to_numpy()
    P_[nm] = np.log1p(Mx)
print("панель:", P_["gmv"].shape)

L = 4                                    # лагов недель
tgt_w = P_["gmv"][:, L:]                 # y_{t}
def lags(nm):
    return [P_[nm][:, L - k: -k if k else None] for k in range(1, L + 1)]

def demean(a):
    return a - a.mean(1, keepdims=True)

Y = demean(tgt_w).ravel()
def design(blocks):
    return np.column_stack([demean(b).ravel() for b in blocks])

def r2(Xd, Yv):
    Xd = np.column_stack([Xd, np.ones(len(Xd))])
    beta, *_ = np.linalg.lstsq(Xd, Yv, rcond=None)
    res = Yv - Xd @ beta
    return 1 - res.var() / Yv.var()

models = {
    "только лаги GMV":            lags("gmv"),
    "+ лаги корзин":              lags("gmv") + lags("cart"),
    "+ лаги запросов":            lags("gmv") + lags("srch"),
    "+ лаги активных дней":       lags("gmv") + lags("act"),
    "всё вместе":                 lags("gmv") + lags("cart") + lags("srch") + lags("act"),
}
scores = {k: r2(design(v), Y) for k, v in models.items()}
b = scores["только лаги GMV"]
for k, v in scores.items():
    print(f"{v:.4f}  (Δ {v-b:+.4f})   {k}")

fig = go.Figure(bars(list(scores), [scores[k] - b for k in scores], slot=0,
                     text=[f"{scores[k]-b:+.4f}" for k in scores]))
fig.update_yaxes(title="прирост within-R² к модели «только лаги GMV»")
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Granger-стиль: что добавляет информацию о будущем GMV",
     sub=f"{GR_N:,} пользователей, внутренняя вариация", h=560, w=980)

панель: (30000, 59)


0.0157  (Δ +0.0000)   только лаги GMV
0.0244  (Δ +0.0087)   + лаги корзин
0.0263  (Δ +0.0106)   + лаги запросов
0.0245  (Δ +0.0088)   + лаги активных дней
0.0280  (Δ +0.0123)   всё вместе


**Оговорки, без которых выводы неверны.** Granger-причинность — это про предсказание,
а не про механизм: общий ненаблюдаемый драйвер (сезон, потребность, кампания) даст тот же
результат. Внутри-пользовательская вариация снимает статические конфаундеры, но не
динамические. Для соревнования вывод всё равно операционный: **если лаги корзин/запросов
дают заметный прирост within-R² сверх лагов GMV — не-GMV-сигналы обязаны быть в модели**;
если прирост около нуля — они дублируют историю покупок, и на них не стоит тратить ёмкость.

## 12. Выводы и план

Сводка того, что данные говорят про конструкцию решения.

In [61]:
summary = {
    "строк / пользователей":        f"{df.height:,} / {N_USERS:,}",
    "заполненность сетки user×day": f"{100*df.height/(N_USERS*N_DAYS):.1f}%",
    "доля дней с gmv>0":            f"{100*(df['gmv']>0).mean():.1f}% строк",
    "нулевой таргет (30д)":         f"{100*(yv==0).mean():.1f}%",
    "макс. recency в выборке":      f"{up['recency'].max()} дней (правило отбора, §3.1)",
    "Джини по GMV за историю":      f"{gini:.3f}",
    "RMSLE «все нули»":             f"{rmsle(yv, np.zeros_like(yv)):.4f}",
    "RMSLE sample_submit-логики":   f"{rmsle(yv, g30):.4f}  (= «прошлые 30д»)",
    "лучший бейзлайн RMSLE":        f"{min(v for _, v in res):.4f} — {min(res, key=lambda t: t[1])[0]}",
    "якорей по 168д до валидации":  f"{len(anchors)} (шаг {STRIDE}д), непересекающихся {len(disjoint)}",
    "оптимальное сжатие log1p":     f"a ≈ {a_best:.2f}",
}
for k, v in summary.items():
    print(f"{k:<32} {v}")

строк / пользователей            30,631,006 / 250,000
заполненность сетки user×day     30.0%
доля дней с gmv>0                15.5% строк
нулевой таргет (30д)             45.9%
макс. recency в выборке          29 дней (правило отбора, §3.1)
Джини по GMV за историю          0.700
RMSLE «все нули»                 3.2036
RMSLE sample_submit-логики       2.1951  (= «прошлые 30д»)
лучший бейзлайн RMSLE            1.8361 — expm1(0.70·log1p(90д/3))
якорей по 168д до валидации      8 (шаг 30д), непересекающихся 2
оптимальное сжатие log1p         a ≈ 0.70


### Что из этого следует

**1. Выборка условна на активности — это влияет на CV сильнее всего (§3.1).**
Максимальный `recency` ровно 29 дней, и все 250 000 пользователей имеют событие в каждом
из трёх последних 30-дневных блоков. Значит на каждом обучающем якоре нужно применять
**тот же фильтр** (`selected_users(A)`), иначе трейн содержит популяцию, которой в тесте
не существует, и `P(y>0)` уезжает. Это дешёвая правка с прямым эффектом на лидерборд.

**2. `sample_submit.csv` — это факт предыдущего 30-дневного окна (§4.1).**
Он же бейзлайн «прошлые 30 дней» и одновременно ориентир по ожидаемому распределению
сабмита (доля ненулевых, средний `log1p`). Отклонение вашего прогноза от этих чисел —
сигнал проверить калибровку.

**3. Hurdle обязателен.** Около половины таргета — нули (точное число в сводке выше),
а условное распределение при `y>0` почти логнормально (§4, §8). Схема:
`p̂ = P(y>0 | x)` × `m̂ = E[log1p(y) | x, y>0]`, а финальное преобразование подбирается
**под RMSLE на валидации**, а не берётся как `p̂ · expm1(m̂)`. §4 показал главное:
оптимальное сжатие в log-шкале всегда `a < 1` — модель обязана выучить регрессию к среднему.

**4. Калибровку не ломать.** Дублирование позитивов (в GA Revenue до 60×) сдвигает `p̂`.
Если сэмплируете — возвращайте prior: `p = p_s / (p_s + (1-p_s)·w)`. Лучше обойтись
весами классов, а качество мерить в RMSLE, а не в AUC.

**5. Recency работает сильнее, чем в GA Revenue.** Там между историей и таргетом было
46 дней cooling period, у нас окно начинается на следующий день. При этом recency у нас
ограничен сверху 29 днями, так что весь сигнал сидит в тонких различиях внутри месяца:
`r_act`, `r_ord`, `r_gmv` + плотность за 7/14/30 дней (§8).

**6. Многооконная сборка.** 168-дневное окно признаков + 30-дневный таргет, якоря с шагом
30 дней (§5), с фильтром отбора на каждом. Валидация — **только по времени**, на якоре
`2026-01-14`, максимально похожем на боевой.

**7. Разреженность не дозаполнять.** Все агрегации считаются по прореженной панели через
целочисленный `d` (§0, §6) — полная сетка в 102 млн строк не нужна ни разу.

**8. Не всё стоит усилий.** Фурье-гармоники (§9) дают меньше, чем простой профиль дня
недели; токен-энтропия (§10) — слабый, но дешёвый признак. Основную ценность несут
RFM + оконные суммы + тренды. Granger-блок (§11.3) показывает, сколько ровно добавляют
не-GMV-сигналы сверх истории покупок — используйте это число, чтобы решить, стоит ли
их обвешивать признаками. Нейросеть по токенам (§10) имеет смысл как второй,
независимый по природе ошибок источник для блендинга.

**9. Перед сабмитом.** Прогноз строится на якоре `2026-02-13` тем же `build_features`,
что и валидация; клипать отрицательные значения самому; ровно 250 000 `user_id`;
сверить долю ненулевых и средний `log1p` с `sample_submit.csv`.